# Persiapan

## Pemasangan Pustaka

Dijalankan perintah *shell* pip install berargumen -q dari dalam *notebook* untuk memasang pustaka duckdb, beautifulsoup4, lxml, requests, pyarrow, dan pandas sesuai langkah kedua bagian G modul. Argumen -q membuat keluaran pemasangan diringkas sehingga hanya pesan galat yang ditampilkan. Pustaka lxml dipasang bersama beautifulsoup4 karena parser "xml" pada BeautifulSoup yang dipakai di H-4 bergantung padanya, sedangkan pyarrow menjadi mesin baca dan tulis Parquet bagi pandas mulai H-6.

In [1]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

## Menghubungkan Google Drive

Dilakukan impor terhadap modul shutil, kelas Path dari pathlib, dan modul drive dari google.colab. Konstanta TITIK_MOUNT_DRIVE bernilai /content/drive, sedangkan DIR_SIMPAN dibentuk sebagai objek Path yang menunjuk folder MyDrive/BigData/Praktikum5, yaitu folder penyimpanan permanen yang sudah disiapkan di Google Drive. *Tuple* LAPISAN_DIARSIPKAN memuat tiga nama folder yang wajib dikumpulkan menurut identitas praktikum, yaitu lapisan_mentah, lapisan_terkurasi, dan siap_pakai.

Didefinisikan fungsi arsipkan_lapisan(nama_folder, direktori_tujuan=DIR_SIMPAN) yang memanggil shutil.make_archive() berformat zip dengan root_dir="." dan base_dir=nama_folder, sehingga isi arsip diawali nama folder lapisannya, lalu me-*return* jalur berkas zip sebagai objek Path. Fungsi ini baru dipanggil pada sel terakhir *notebook* setelah seluruh lapisan terisi. Google Drive disambungkan ke *runtime* melalui drive.mount(TITIK_MOUNT_DRIVE) agar artefak hasil tidak hilang ketika *runtime* berakhir, lalu DIR_SIMPAN dibuat dengan mkdir() berargumen parents=True dan exist_ok=True sehingga sel aman dijalankan berulang kali. Jalur folder dan daftar isinya dicetak sebagai verifikasi bahwa penyambungan berhasil.

In [2]:
import shutil
from pathlib import Path

from google.colab import drive

TITIK_MOUNT_DRIVE = "/content/drive"
DIR_SIMPAN = Path(TITIK_MOUNT_DRIVE, "MyDrive", "BigData", "Praktikum5")
LAPISAN_DIARSIPKAN = ("lapisan_mentah", "lapisan_terkurasi", "siap_pakai")


def arsipkan_lapisan(nama_folder, direktori_tujuan=DIR_SIMPAN):
    """Kompres satu folder lapisan menjadi berkas zip di direktori tujuan.

    Folder dibaca dari direktori kerja runtime, dan jalur berkas zip yang
    terbentuk dikembalikan sebagai objek Path.
    """
    jalur_tanpa_ekstensi = Path(direktori_tujuan, nama_folder)
    jalur_zip = shutil.make_archive(
        str(jalur_tanpa_ekstensi), "zip", root_dir=".", base_dir=nama_folder
    )
    return Path(jalur_zip)


drive.mount(TITIK_MOUNT_DRIVE)
DIR_SIMPAN.mkdir(parents=True, exist_ok=True)
isi_folder = sorted(item.name for item in DIR_SIMPAN.iterdir())

print("Folder penyimpanan:", DIR_SIMPAN)
print("Isi saat ini      :", isi_folder)

Mounted at /content/drive
Folder penyimpanan: /content/drive/MyDrive/BigData/Praktikum5
Isi saat ini      : []


# H. Langkah Praktikum

## H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal

Dilakukan impor terhadap modul os, re, json, time, zlib, queue, threading, dan collections, kelas Path dari pathlib, modul urllib.request, serta pustaka pandas dengan alias pd. Konstanta URL_DATA menyimpan alamat berkas detik_news_title.csv pada repositori GitHub ibamibrahim/dataset-judul-berita-indonesia, lalu berkas tersebut diunduh melalui urllib.request.urlretrieve() hanya apabila *if statement* atas os.path.exists() menyatakan berkas belum ada, sehingga sel tidak mengunduh ulang ketika dijalankan kembali. Berkas dibaca ke df_sumber dengan pd.read_csv(), kemudian dicetak ukuran berkas dalam MB dari os.path.getsize() yang dibagi 1e6 dan dibulatkan satu desimal, jumlah baris dan kolom melalui shape, serta daftar nama kolom.

Bagian kedua mengimpor modul http.server, urllib.parse, dan html, lalu menetapkan HALAMAN_UKURAN bernilai 100 judul per halaman. Fungsi jalankan_portal(df, port=8000) mengubah DataFrame menjadi list *dictionary* melalui to_dict("records") dan menghitung n_hal sebagai pembulatan ke atas dari jumlah baris dibagi HALAMAN_UKURAN. Di dalamnya didefinisikan kelas Handler turunan http.server.BaseHTTPRequestHandler yang mematikan log melalui log_message(), mengirim status, *header* Content-Type dan Content-Length, serta isi halaman berpengkodean UTF-8 melalui method \_kirim(kode, teks, tipe), dan melayani permintaan melalui do_GET(). Pada do_GET(), *if statement* atas u.path melayani /robots.txt dengan aturan Disallow: /arsip-privat/, melayani /arsip-privat/ dengan pesan larangan, serta melayani /berita dengan membaca parameter hal melalui parse_qs(). Halaman di luar rentang 1 sampai n_hal dibalas status 404, sedangkan halaman yang sah disusun dari satu elemen article.item per judul yang memuat h2.judul berisi tautan, span.kategori, dan elemen time beratribut datetime berformat ISO hasil pd.to_datetime() dengan format="%m/%d/%Y". Seluruh teks dimasukkan melalui html.escape(), dan jalur lain dibalas status 404. Argumen kedua send_header() untuk Content-Type ditulis tipe sesuai modul, karena versi sebelumnya mengirim variabel data, yaitu isi halaman bertipe *bytes*, sebagai nilai *header*. Server ThreadingHTTPServer pada 127.0.0.1 dijalankan di *thread* daemon melalui threading.Thread(target=server.serve_forever, daemon=True), lalu objek server dan n_hal yang di-*return* di-*assign* ke server dan N_HALAMAN. Konstanta BASE ditetapkan bernilai http://127.0.0.1:8000, kemudian alamat portal dan jumlah halaman dicetak.

In [3]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd


URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
  urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas: ", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom: ", df_sumber.shape)
print("Kolom: ", list(df_sumber.columns))

import http.server, urllib.parse, html

HALAMAN_UKURAN = 100

def jalankan_portal(df, port=8000):
  """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML. """
  baris = df.to_dict("records")
  n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN


  class Handler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *args):
      pass
    def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
      data = teks.encode("utf-8")
      self.send_response(kode)
      self.send_header("Content-Type", tipe)
      self.send_header("Content-Length", str(len(data)))
      self.end_headers()
      self.wfile.write(data)
    def do_GET(self):
      u = urllib.parse.urlparse(self.path)
      if u.path == "/robots.txt":
        self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
      elif u.path == "/arsip-privat/":
        self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
      elif u.path == "/berita":
        q = urllib.parse.parse_qs(u.query)
        hal = int(q.get("hal", ["1"])[0])
        if not 1 <= hal <= n_hal:
          return self._kirim(404, "<p>Halaman tidak ada</p>")
        potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
        isi = []
        for r in potong:
          iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
          isi.append(
              '<article class="item">'
              f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
              f'<span class = "kategori">{html.escape(r["category"])}</span>'
              f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
              '</article>'
          )
        self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                          f'<nav data-total="{n_hal}"></nav></body></html>')
      else:
        self._kirim(404, "<p>Tidak ditemukan</p>")

  server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
  threading.Thread(target=server.serve_forever, daemon=True).start()
  return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)



Ukuran berkas:  16.1 MB
Baris, kolom:  (91017, 4)
Kolom:  ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


## H-3. Crawling Portal Lokal: Membaca robots.txt

Dilakukan impor terhadap pustaka requests, modul urllib.robotparser, dan kelas BeautifulSoup dari bs4. Konstanta UA berisi identitas KelasBigData-UNAND/1.0 beserta keterangan praktikum dan kontak dosen sebagai nilai User-Agent. Fungsi baca_robots(base) mengambil robots.txt dari base melalui requests.get() dengan *header* User-Agent dan timeout=10, membentuk objek RobotFileParser yang mem-*parse* isi berkas per baris melalui parse(r.text.splitlines()), lalu me-*return* objek tersebut. Fungsi dipanggil terhadap BASE dan hasilnya di-*assign* ke robots. Dijalankan *for loop* atas dua jalur, yaitu /berita?hal=1 dan /arsip-privat/, dan pada setiap iterasi hasil robots.can_fetch(UA, BASE + jalur) dicetak sebagai jawaban boleh atau tidaknya jalur tersebut di-*crawl*.

In [4]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
  r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
  rp = urllib.robotparser.RobotFileParser()
  rp.parse(r.text.splitlines())
  return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
  boleh = robots.can_fetch(UA, BASE + jalur)
  print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


## H-3. Parser Halaman dan Crawler Portal Lokal

Didefinisikan fungsi parse_halaman(teks_html) yang mem-*parse* HTML satu halaman dengan BeautifulSoup berparser html.parser, lalu menjalankan *for loop* atas setiap elemen hasil select("article.item"). Pada setiap elemen, tautan diambil dengan select_one("h2.judul a"), kemudian satu *dictionary* berisi judul dari get_text(strip=True), url dari atribut href, kategori dari span.kategori, dan tanggal dari atribut datetime milik elemen time ditambahkan ke list hasil yang di-*return* di akhir fungsi. Atribut datetime dipakai karena formatnya ISO, sedangkan teks tanggal berformat bulan/hari/tahun dapat tertukar dengan hari/bulan/tahun.

Fungsi crawl_iter(base, halaman, delay=0.0, ua=UA) membaca robots.txt melalui baca_robots(), membentuk requests.Session() dengan *header* User-Agent bernilai ua, lalu menjalankan *for loop* atas nomor halaman. Pada setiap iterasi, URL /berita?hal=<nomor> disusun, dan *if statement* atas can_fetch() melewati halaman yang dilarang dengan mencetak pesan lalu continue. Pengambilan halaman dicoba paling banyak tiga kali melalui *for loop* range(3) berisi blok *try-except*, yaitu sesi.get() dengan timeout=10 dan raise_for_status() yang diakhiri break bila berhasil, sedangkan *exception* requests.RequestException ditangani dengan time.sleep(0.5 \* (percobaan + 1)) sehingga jeda bertambah pada setiap percobaan. Klausa *else* milik *for loop* tersebut hanya berjalan apabila ketiga percobaan gagal, lalu mencetak pesan gagal dan melanjutkan ke halaman berikutnya. Halaman yang berhasil diubah dengan parse_halaman() dan hasilnya dikeluarkan satu per satu melalui yield from, diikuti time.sleep(delay). Fungsi diuji pada halaman 1 sampai 3 melalui range(1, 4), hasilnya diubah menjadi list contoh, lalu dicetak jumlah judul dan tiga *dictionary* pertama.

In [5]:
def parse_halaman(teks_html):
  """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
  sup = BeautifulSoup(teks_html, "html.parser")
  hasil = []
  for a in sup.select("article.item"):
    link = a.select_one("h2.judul a")
    hasil.append({
        "judul": link.get_text(strip=True),
        "url": link["href"],
        "kategori": a.select_one("span.kategori").get_text(strip=True),
        "tanggal": a.select_one("time")["datetime"],
    })
  return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
  """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda"""
  rp = baca_robots(base)
  sesi = requests.Session()
  sesi.headers["User-Agent"] = ua
  for h in halaman:
    url = f"{base}/berita?hal={h}"
    if not rp.can_fetch(ua, url):
      print("Dilewati (robots.txt): ", url)
      continue
    for percobaan in range(3):
      try:
        r = sesi.get(url, timeout=10)
        r.raise_for_status()
        break
      except requests.RequestException as e:
        time.sleep(0.5 * (percobaan + 1))
    else:
      print("Gagal setelah 3 percobaan: ", url)
      continue
    yield from parse_halaman(r.text)
    time.sleep(delay)
contoh = list(crawl_iter(BASE, range(1,4)))
print("Judul dari 3 halaman: ", len(contoh))
for c in contoh[:3]:
  print(c)

Judul dari 3 halaman:  300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


## H-4. Langkah 1: Parser Sitemap Berita

Kelas BeautifulSoup dan fungsi urlparse dari urllib.parse diimpor, lalu konstanta POLA_ARTIKEL dibentuk melalui re.compile() dengan pola https?://([\w-]+)\\.detik\\.com/.+/d-(\d+)/ yang menangkap subdomain pada grup pertama dan nomor artikel pada grup kedua. Didefinisikan fungsi parse_sitemap_news(xml_data) yang mem-*parse* XML dengan BeautifulSoup berparser "xml", lalu menjalankan *for loop* atas setiap elemen hasil find_all("url"). Pada setiap entri, elemen loc, title, dan publication_date dicari dengan find() tanpa awalan *namespace* news: karena awalan tersebut diabaikan saat pencarian tag. *If statement* melewati entri yang tidak memiliki loc atau judul melalui continue. Entri yang lengkap ditambahkan ke list hasil sebagai *dictionary* berisi judul yang spasinya dirapikan dengan re.sub(r"\s+", " ", …), url, kategori dari bagian pertama urlparse(url).netloc yang dipisah pada titik, id berupa int(m.group(2)) apabila POLA_ARTIKEL cocok dan None apabila tidak, tanggal berupa sepuluh karakter pertama publication_date apabila elemennya ada, serta kata_kunci dari elemen keywords atau *string* kosong. Tanggal hanya diambil sepuluh karakter karena tanda zona waktu pada sitemap diragukan menurut catatan zona waktu pada modul. List hasil di-*return* di akhir fungsi.

In [6]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": urlparse(url).netloc.split(".")[0], # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

## H-4. Langkah 2: Uji Parser tanpa Internet

Konstanta CONTOH_XML berisi *string* XML berelemen urlset dengan *namespace* sitemap dan news, memuat tiga entri artikel news.detik.com hasil uji dosen tanggal 29 Oktober 2026 yang isinya berbentuk CDATA, dan hanya entri pertama yang memiliki news:keywords. Dijalankan *for loop* atas hasil parse_sitemap_news(CONTOH_XML), lalu pada setiap entri dicetak *dictionary* yang hanya memuat kunci judul, kategori, id, dan tanggal melalui *dictionary comprehension*. Setelah itu dicetak kata_kunci milik entri pertama, kemudian dicetak hasil perbandingan == antara keluaran fungsi untuk masukan *bytes* hasil encode("utf-8") dan keluaran untuk masukan *string*, sebagai pemeriksaan bahwa fungsi menerima kedua tipe masukan.

In [7]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi, mk, hukum, kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi, mk, hukum, kepala daerah
Sama bila masukan berupa bytes: True


## H-4. Langkah 3: Fungsi Pengambilan yang Sopan

Modul urllib.robotparser diimpor kembali, lalu ditetapkan empat konstanta pengaman, yaitu UA_DETIK berisi identitas BigData/1.0 beserta keterangan praktikum dan kolom kontak yang menurut modul diisi email mahasiswa, DELAY bernilai 2,0 detik jeda antar-*request*, BATAS_PER_BAGIAN bernilai 7 judul per sitemap, dan BATAS_TOTAL bernilai 20 judul untuk seluruh *crawling*. List SITEMAP_DIPILIH memuat tiga alamat sitemap berita, yaitu news.detik.com/berita, inet.detik.com/news, dan sport.detik.com/basket, sedangkan *dictionary* kosong \_robots disiapkan sebagai penampung RobotFileParser per *host*. Fungsi boleh_diambil(url, ua=UA_DETIK) mengambil *host* dari urlparse(url).netloc, lalu *if statement* membuat RobotFileParser baru dengan set_url() pada https://<host>/robots.txt dan read() hanya apabila *host* belum ada di \_robots, sehingga robots.txt setiap *host* diunduh sekali, kemudian fungsi me-*return* hasil can_fetch(ua, url).

Fungsi ambil_xml(url, sesi, delay=DELAY) melempar PermissionError apabila boleh_diambil() bernilai False, mengirim sesi.get() dengan timeout=20, melempar RuntimeError beserta pesan untuk berhenti apabila status balasan termasuk 403 atau 429, dan memanggil raise_for_status() untuk status galat lain. Fungsi kemudian menunggu selama delay melalui time.sleep() dan me-*return* r.content bertipe *bytes* sehingga pengkodean dibaca dari deklarasi XML. Fungsi crawl_detik() membentuk requests.Session() dengan *header* User-Agent bernilai UA_DETIK dan list kosong hasil, lalu menjalankan *for loop* atas SITEMAP_DIPILIH yang dihentikan melalui break apabila jumlah hasil sudah mencapai BATAS_TOTAL. Pada setiap iterasi, keluaran parse_sitemap_news() atas ambil_xml() dipotong melalui *slicing* [:BATAS_PER_BAGIAN] dan digabungkan ke hasil, lalu fungsi me-*return* hasil[:BATAS_TOTAL]. Sel ini hanya mendefinisikan fungsi sehingga belum ada *request* yang dikirim ke detik.com.

In [8]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: 2411531006_daffa@student.unand.ac.id)"
DELAY = 2.0               # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7      # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20          # batas seluruh crawling

SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}

def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content  # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

## H-4. Langkah 4: Menjalankan dan Memeriksa Hasil Crawling

Fungsi crawl_detik() dijalankan di dalam blok *try-except*, dan *exception* apa pun ditangani dengan meng-*assign* list kosong ke sampel serta mencetak nama kelas *exception* beserta pesannya, sehingga kegagalan jaringan tidak menghentikan Run all. *If statement* berikutnya hanya berjalan apabila sampel berisi data. Folder contoh_crawl dibuat dengan mkdir(exist_ok=True), lalu setiap entri ditulis sebagai satu baris JSON ke contoh_crawl/detik_sitemap.jsonl melalui json.dumps() berargumen ensure_ascii=False dengan pengkodean UTF-8. Folder ini terpisah dari lapisan_mentah sehingga isi lapisan mentah dan angka acuan pada langkah berikutnya tidak berubah. Sampel diubah menjadi DataFrame cek, lalu dicetak jumlah judul, jumlah judul kosong dari str.strip() yang sama dengan *string* kosong, jumlah URL duplikat melalui duplicated(), jumlah tanggal kosong melalui isna(), sebaran kategori melalui value_counts().to_dict(), dan tiga baris pertama kolom judul, kategori, dan tanggal melalui to_string(index=False).

In [9]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul :", len(cek))
    print("Judul kosong    :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat    :", cek["url"].duplicated().sum())
    print("Tanggal kosong  :", cek["tanggal"].isna().sum())
    print("Kategori        :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul : 20
Judul kosong    : 0
URL duplikat    : 6
Tanggal kosong  : 0
Kategori        : {'news': 7, '': 7, 'sport': 6}
                                                                    judul kategori    tanggal
    Jokowi, Gibran hingga Kaesang Hadiri Royal Wedding Pura Mangkunegaran     news 2026-11-06
             Kebakaran Resto di Blok M Square Diduga Dipicu Kebocoran Gas     news 2026-11-06
Sespimmen Angkatan 67 Gelar Baksos, Salurkan Sembako-Perlengkapan Sekolah     news 2026-11-06


## H-5. Channel ala Flume ke lapisan_mentah

Konstanta RAW ditetapkan sebagai Path("lapisan_mentah") dan foldernya dibuat dengan mkdir(exist_ok=True), lalu setiap berkas .jsonl lama di dalamnya dihapus melalui *for loop* atas RAW.glob("\*.jsonl") dan unlink() sehingga hasil tidak bercampur dengan eksekusi sebelumnya. Channel dibentuk sebagai queue.Queue(maxsize=2000), STOP dibentuk dari object() sebagai penanda akhir aliran, dan *dictionary* statistik disiapkan dengan kunci max_antrean, masuk, keluar, dan file yang bernilai awal 0. Fungsi source() menjalankan *for loop* atas crawl_iter(BASE, range(1, N_HALAMAN + 1)), memasukkan setiap event ke channel melalui put() yang menunggu ketika antrean penuh, menambah statistik masuk, dan memperbarui max_antrean dengan max() atas qsize(), lalu memasukkan STOP setelah seluruh halaman selesai.

Fungsi sink(ukuran_batch=5000) menyiapkan list batch, nomor berkas bernilai 0, dan fungsi dalam tulis() yang memakai nonlocal untuk menaikkan nomor, menulis seluruh isi batch sebagai baris JSON ke RAW/batch_<nomor empat digit>.jsonl, mencatat nomor ke statistik file, dan mengosongkan batch. Sink menjalankan *while loop* yang mengambil event dengan get(), keluar melalui break apabila event adalah STOP, dan memanggil tulis() setiap kali panjang batch mencapai ukuran_batch, lalu *if statement* setelah *loop* menulis sisa batch yang belum penuh. Waktu mulai dicatat dengan time.perf_counter(), source dan sink dijalankan pada dua *thread* terpisah melalui threading.Thread(), start(), dan join(), lalu selisih waktunya di-*assign* ke waktu. Dicetak jumlah event yang masuk channel dan yang ditulis sink dengan pemisah ribuan, jumlah berkas batch beserta tiga nama berkas pertama, hasil pemeriksaan antrean maksimum tidak melebihi 2.000, total ukuran berkas mentah dalam MB, dan waktu *crawl* untuk N_HALAMAN halaman.

In [10]:
RAW = Path("lapisan_mentah")
RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)  # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []

    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev)
        statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source)
t_snk = threading.Thread(target=sink)
t_src.start()
t_snk.start()
t_src.join()
t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 51.6 detik untuk 911 halaman


## H-6. Aturan Kualitas Data Mentah

Seluruh berkas JSONL pada RAW dibaca berurutan melalui *list comprehension* atas sorted(RAW.glob("\*.jsonl")) dengan pd.read_json() berargumen lines=True, lalu digabung menjadi DataFrame mentah melalui pd.concat() berargumen ignore_index=True dan jumlah barisnya dicetak. *Set* KATEGORI_DIKENAL memuat sembilan kategori dataset, yaitu finance, food, health, hot, inet, news, oto, sport, dan travel. Fungsi aturan_kualitas(d) menghitung jml_kata dari jumlah token hasil str.split() atas judul yang nilai kosongnya diisi *string* kosong, lalu me-*return* *dictionary* berisi enam Series *boolean*, yaitu judul tidak kosong, judul 3-30 kata melalui between(3, 30), kategori dikenal melalui isin(KATEGORI_DIKENAL), tanggal valid melalui pd.to_datetime() berformat %Y-%m-%d dengan errors="coerce" yang diikuti notna(), url artikel valid melalui str.contains() berpola /d-\d+ dengan regex=True, dan judul tidak duplikat melalui negasi duplicated(keep=False). Dijalankan *for loop* atas pasangan nama aturan dan Series hasilnya, lalu dicetak jumlah dan persentase baris yang gagal dari (~lulus).sum() dan (~lulus).mean() yang dikalikan 100.

Keputusan pra-pemrosesan pada sel ini ada dua, dan keduanya mengikuti modul. Pertama, judul dinilai layak apabila berisi 3 sampai 30 kata, sehingga judul di luar rentang itu diperlakukan sebagai kemungkinan salah ambil, dengan konsekuensi judul sah yang sangat pendek ikut tertandai gagal. Kedua, aturan duplikat memakai keep=False sehingga seluruh salinan judul dihitung gagal, bukan hanya salinan kedua dan seterusnya, sehingga angka gagalnya menggambarkan banyaknya baris yang terlibat duplikasi. Sel ini hanya memeriksa dan tidak mengubah mentah, sedangkan pembuangan baris dilakukan pada sel berikutnya.

In [11]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))], ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"   : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"      : jml_kata.between(3, 30),
        "kategori dikenal"     : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"        : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"    : d["url"].str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat" : ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"  {nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
  judul tidak kosong       gagal:      0 ( 0.00%)
  judul 3-30 kata          gagal:     78 ( 0.09%)
  kategori dikenal         gagal:      0 ( 0.00%)
  tanggal valid            gagal:      0 ( 0.00%)
  url artikel valid        gagal:  5,640 ( 6.20%)
  judul tidak duplikat     gagal:    235 ( 0.26%)


## H-6. Pembersihan Data

DataFrame mentah disalin ke bersih melalui copy(), lalu spasi berlebih pada judul dirapikan dengan str.replace(r"\s+", " ", regex=True) yang diikuti str.strip(). Baris berjudul ganda dibuang dengan drop_duplicates(subset="judul", keep="first") setelah spasi dirapikan, sehingga judul yang hanya berbeda spasi dianggap sama. Selanjutnya, bersih disaring pada judul yang jumlah katanya berada di dalam between(3, 30) dan hasilnya disalin dengan copy(). Kolom tanggal diubah menjadi datetime melalui pd.to_datetime() berformat eksplisit %Y-%m-%d, kolom bulan dibentuk dari dt.strftime("%Y-%m"), dan kolom id diekstrak dari pola /d-(\d+) pada url melalui str.extract() lalu di-*cast* ke tipe Int64 yang mendukung nilai kosong. Terakhir, url di-*assign* ulang melalui where() sehingga hanya url yang memuat pola /d-\d+ yang dipertahankan, sedangkan url lain menjadi kosong. Dicetak jumlah baris setelah pembersihan dan jumlah baris yang dibuang dari selisih panjang mentah dan bersih.

Keputusan pra-pemrosesan pada sel ini ada empat, dan seluruhnya mengikuti modul. Pertama, duplikat didefinisikan pada kolom judul saja dan kemunculan pertama dipertahankan, karena analisis berikutnya menghitung kata pada judul sehingga judul yang sama akan menggandakan frekuensi kata, dengan risiko dua berita berbeda yang kebetulan berjudul sama ikut tergabung. Kedua, judul di luar 3 sampai 30 kata dibuang, bukan diperbaiki, karena isi judul yang benar tidak dapat dipulihkan dari data yang tersedia. Ketiga, baris kategori inet yang url-nya rusak tidak dibuang, melainkan url-nya dikosongkan, sehingga judulnya tetap ikut dianalisis dan url-nya tidak terpakai sebagai kunci, dengan konsekuensi kolom id juga kosong untuk seluruh baris tersebut. Keempat, format tanggal ditulis eksplisit agar urutan bulan dan hari tidak ditebak oleh pandas, sebagaimana kekeliruan yang dialami pada Praktikum 2.

In [12]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))  # kosong bila url rusak

print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


## H-6. Parquet Berpartisi dan Partition Pruning

Dilakukan impor terhadap modul shutil dan pyarrow.dataset dengan alias ds. Fungsi simpan_terpartisi(data, kolom_partisi, folder) menghapus folder lama melalui shutil.rmtree() berargumen ignore_errors=True, menulis data sebagai Parquet berpartisi melalui to_parquet() dengan partition_cols=kolom_partisi dan index=False, lalu me-*return* jumlah berkas .parquet hasil rglob() beserta total ukurannya dalam MB. Dijalankan *for loop* atas tiga pilihan kolom partisi, yaitu ["kategori"], ["bulan"], dan ["kategori", "bulan"], dan setiap pilihan ditulis ke folder uji_partisi lalu jumlah berkas dan ukurannya dicetak, kemudian folder uji tersebut dihapus. Konstanta TERKURASI ditetapkan bernilai lapisan_terkurasi, lalu bersih disimpan ke sana dengan partisi kategori dan hasilnya dicetak. Folder tersebut dibaca kembali sebagai ds.dataset() berformat parquet dengan partitioning="hive", lalu jumlah seluruh fragmen dari get_fragments() dibandingkan dengan jumlah fragmen yang tersisa setelah filter ds.field("kategori") == "finance" sebagai bukti *partition pruning*.

Keputusan pra-pemrosesan pada sel ini adalah pemilihan kategori sebagai kolom partisi lapisan terkurasi, mengikuti modul. Pertanyaan analisis menyangkut perbandingan antar-kategori sehingga filter pada kategori dapat langsung melewati berkas yang tidak diperlukan, dan berkas setiap kategori nantinya menjadi input split MapReduce pada H-9. Konsekuensinya, ukuran setiap partisi tidak seimbang karena jumlah judul per kategori berbeda jauh, sedangkan penambahan bulan sebagai kolom partisi hanya menambah jumlah berkas kecil tanpa keuntungan bagi pertanyaan tersebut.

In [13]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori finance


## H-7. Query DuckDB dan Bentuk Dokumen

Pustaka duckdb diimpor dan koneksi dalam memori dibentuk melalui duckdb.connect(), lalu *view* berita dibuat dengan CREATE VIEW atas read_parquet() pada pola lapisan_terkurasi/\*/\*.parquet berargumen hive_partitioning=true sehingga kolom kategori dibaca dari nama folder. Query pertama menghitung jumlah judul per kategori dengan COUNT(\*), GROUP BY kategori, dan ORDER BY jumlah DESC, sedangkan query kedua menghitung judul yang memuat kata corona per bulan melalui lower(judul) LIKE '%corona%' dengan GROUP BY bulan dan ORDER BY bulan. Kedua hasil diubah ke DataFrame dengan df() dan dicetak melalui to_string(index=False). Setelah itu, baris pertama bersih yang diambil melalui iloc[0] diubah menjadi *dictionary* dokumen berisi id yang di-*cast* ke int, judul, kategori, tanggal berbentuk *string* dari date(), serta list kata hasil re.findall(r"[a-z]+") atas judul berhuruf kecil. Dokumen dicetak sebagai JSON melalui json.dumps() berargumen ensure_ascii=False dan indent=2.

In [14]:
import duckdb

con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


## H-8. Mini MapReduce: Kasus Kantin

List kasir berisi tiga list struk, yaitu milik kasir A, B, dan C, dengan setiap struk berbentuk *string* menu dan jumlah yang dipisahkan koma, sesuai permainan kartu pada H-1. Fungsi map_fn(baris) memisahkan *string* dengan split(",") dan me-*return* *tuple* (menu, int(jumlah)), sedangkan reduce_fn(menu, daftar_nilai) me-*return* *tuple* menu beserta sum() dari daftar nilainya. Tahap map dijalankan melalui *list comprehension* bersarang sehingga hasil_map berisi satu list pasangan per kasir, lalu hasilnya dicetak per kasir melalui *for loop* atas zip("ABC", hasil_map). Tahap shuffle mengumpulkan nilai ke *dictionary* shuffle berdasarkan menu melalui setdefault(menu, []).append(jumlah) di dalam *for loop* bersarang, lalu tahap reduce membentuk *dictionary* hasil dari reduce_fn() atas setiap pasangan pada shuffle.items() dan mencetaknya bersama total porsi.

Didefinisikan pula fungsi combiner(pasangan) yang menjumlahkan nilai per menu di sisi setiap kasir memakai collections.Counter() dan me-*return* list pasangan hasil items(). Fungsi ini diterapkan pada setiap list di hasil_map dan hasilnya di-*assign* ke setelah, lalu dicetak jumlah pasangan yang dikirim tanpa combiner dan dengan combiner dari penjumlahan panjang setiap list.

In [15]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],        # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],        # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"],   # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


## H-9. Word Count Terdistribusi: Partisi sebagai Input Split

*Set* STOPWORD dibentuk dari split() atas *string* berisi kata fungsi bahasa Indonesia yang umum, seperti yang, di, dan, ke, dari, ini, dan itu, beserta singkatan percakapan seperti yg dan dgn. Fungsi tokenisasi(teks) me-*return* seluruh rangkaian huruf a sampai z dari teks berhuruf kecil melalui re.findall(r"[a-z]+"). Input split ditetapkan sebagai berkas partisi Parquet, yaitu hasil sorted() atas Path(TERKURASI).rglob("\*.parquet") yang di-*assign* ke splits, lalu dicetak jumlah split dan jumlah judul setiap split melalui pd.read_parquet() yang hanya membaca kolom judul.

Fungsi mapper_split(path) membaca kolom judul satu split dan me-*return* list pasangan (kata, 1) untuk setiap kata pada setiap judul melalui *list comprehension* bersarang. Fungsi ini dijalankan atas seluruh splits ke hasil_map, dan total_pasangan dihitung dari penjumlahan panjang setiap list. Tahap shuffle memakai collections.defaultdict(list) untuk mengumpulkan nilai per kata, lalu tahap reduce membentuk *dictionary* hasil_semua dari sum() setiap list nilai. Dicetak jumlah pasangan dari semua mapper, jumlah kata unik, sepuluh kata teratas melalui Counter(hasil_semua).most_common(10), dan sepuluh kata teratas setelah kata pada STOPWORD dikeluarkan melalui *list comprehension* bersyarat. Terakhir, hasil dicek silang dengan collections.Counter atas seluruh token kolom judul pada bersih, lalu dicetak hasil perbandingan == antara keduanya.

Keputusan pra-pemrosesan pada sel ini ada dua, dan keduanya mengikuti modul. Pertama, token hanya terdiri atas huruf a sampai z setelah judul diubah ke huruf kecil, sehingga angka dan tanda baca tidak dihitung sebagai kata, dengan konsekuensi istilah seperti covid-19 terhitung sebagai covid saja. Kedua, STOPWORD hanya berisi kata fungsi umum dan baru diterapkan setelah penghitungan selesai, sehingga hitungan lengkap tetap tersedia untuk pemeriksaan, sedangkan kata khas domain belum tersaring.

In [16]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


## H-10. Combiner dan Jumlah Data yang Berpindah

Kelas Pool diimpor dari multiprocessing. Fungsi mapper_combiner(path) membaca kolom judul satu split, lalu menjumlahkan token setiap judul ke collections.Counter() melalui update() dan me-*return* Counter tersebut, sehingga satu mapper hanya mengirim satu pasangan untuk setiap kata unik. Fungsi jalankan(fn, daftar, proses) mencatat waktu mulai dengan time.perf_counter(), lalu *if statement* menjalankan fn secara berurutan melalui *list comprehension* apabila proses bernilai 1, dan selain itu membagikan pekerjaan ke Pool(proses) melalui pool.map() di dalam blok with, kemudian me-*return* hasil beserta durasinya.

Dicetak jumlah CPU dari os.cpu_count() dan judul tabel, lalu dijalankan *for loop* atas jumlah proses 1 dan 2. Pada setiap jumlah proses, mapper_split dan mapper_combiner masing-masing dijalankan tiga kali melalui *for loop* range(3), durasinya ditampung ke list waktu, lalu dicetak median dari tiga durasi, yaitu sorted(waktu)[1], beserta jumlah pasangan yang dikirim. Hasil eksekusi terakhir mapper_combiner tersimpan di h2, sehingga seluruh Counter di dalamnya dijumlahkan ke total melalui update(), lalu dicetak hasil perbandingan == antara dict(total) dan hasil_semua dari H-9.

In [17]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")

    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.500              825,601
1 proses, dengan combiner         0.547               74,009
2 proses, tanpa combiner          1.207              825,601
2 proses, dengan combiner         1.186               74,009

Hasil dengan combiner sama persis: True


## H-11. Data Skew dan Beban Reducer

Konstanta R ditetapkan bernilai 4 sebagai jumlah reducer. Fungsi partitioner(kata, r=R) me-*return* sisa bagi zlib.crc32() atas kata yang di-encode terhadap r, sehingga nomor reducer suatu kata selalu sama di setiap sesi, berbeda dengan hash() bawaan Python yang nilainya berubah antar-sesi. Fungsi beban(pasangan_per_mapper, r=R) menyiapkan list b berisi r angka nol, lalu menjalankan *for loop* bersarang atas setiap pasangan dari setiap mapper dan menambah satu pada indeks reducer tujuan hasil partitioner(), kemudian me-*return* list tersebut. Fungsi tampil(nama, b) mencetak beban setiap reducer beserta persentasenya terhadap jumlah seluruh pasangan.

Beban empat reducer dicetak untuk hasil_map tanpa combiner dan untuk h2 dengan combiner yang setiap Counter-nya diubah menjadi list pasangan melalui items(). Dijalankan *for loop* atas jumlah reducer 4, 16, dan 64, lalu dicetak persentase beban reducer terberat dari max(b)/sum(b) yang dikalikan 100 berdampingan dengan bagian yang adil, yaitu 100/r. Terakhir, lima kata dengan frekuensi tertinggi pada hasil_semua dicetak bersama jumlah pasangannya, persentasenya terhadap total, dan nomor reducer tujuannya.

In [18]:
R = 4  # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, _ in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"  {nama:<22}", "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

  tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
  dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


## H-12. Menulis Berkas Masukan serta Skrip Mapper dan Reducer

Berkas masukan.tsv ditulis melalui Path.write_text() berpengkodean UTF-8 dengan isi pasangan kategori dan judul dari bersih yang dipisahkan karakter tab, satu baris per judul. Skrip mapper.py ditulis sebagai *string* yang membaca sys.stdin per baris, memisahkan kategori dan judul pada tab pertama melalui split() berargumen maxsplit 1, lalu mencetak setiap kata hasil re.findall(r"[a-z]+") diikuti tab dan angka 1. Skrip reducer.py membaca pasangan kunci dan nilai per baris dengan variabel kunci_sekarang dan total, lalu *if statement* mencetak total kunci sebelumnya setiap kali kunci berganti dan mengatur ulang total ke 0, sedangkan baris berkunci sama hanya menambah total. Setelah *for loop* selesai, total kunci terakhir dicetak oleh *if statement* penutup. Jumlah baris masukan.tsv dihitung melalui *generator expression* dan dicetak.

In [19]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


## H-12. Menjalankan Pipeline dengan Pipe Linux

Dijalankan empat perintah *shell*. Perintah pertama mengalirkan tiga baris pertama masukan.tsv ke mapper.py melalui head -n 3 dan menampilkan lima baris keluarannya. Perintah kedua menjalankan seluruh *pipeline* cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py dengan keluaran ke hasil_pipe.txt, dibungkus time untuk mencatat waktu real, user, dan sys. Perintah sort berperan sebagai tahap shuffle, sedangkan LC_ALL=C membuat pengurutan berdasarkan nilai *byte* seperti pengurutan kunci pada Hadoop. Perintah ketiga menghitung baris hasil_pipe.txt dengan wc -l, dan perintah keempat mengurutkan hasil berdasarkan kolom kedua secara numerik menurun melalui sort berargumen -k2,2nr lalu menampilkan lima baris teratas.

In [20]:
!head -n 3 masukan.tsv | python3 mapper.py | head -n 5
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1

real	0m2.767s
user	0m2.647s
sys	0m0.094s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


## H-12. Membuktikan Hasil Pipe Sama dengan H-9

*Dictionary* kosong hasil_pipe disiapkan, lalu dijalankan *for loop* atas setiap baris hasil_pipe.txt yang dibuka dengan pengkodean UTF-8. Pada setiap iterasi, karakter baris baru dibuang dengan rstrip(), kata dan jumlah dipisahkan pada tab, dan jumlahnya di-*cast* ke int sebelum disimpan dengan kata sebagai kunci. Dicetak hasil perbandingan == antara hasil_pipe dan hasil_semua dari H-9.

In [21]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


## H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

Fungsi mapper_kategori(path) membaca kolom judul satu split dan mengambil nama kategori dari nama folder induk yang dipisah pada tanda sama dengan. *For loop* bersarang atas judul dan token kemudian menambah satu pada Counter berkunci gabungan (kategori, kata) untuk setiap kata yang tidak termasuk STOPWORD, dan Counter tersebut di-*return* per split sehingga combiner terpasang di dalam mapper. Hasil kesembilan mapper dari map(mapper_kategori, splits) dijumlahkan ke Counter gabung melalui update(), lalu dikelompokkan ke defaultdict(list) per_kategori berisi *tuple* (n, kata). Dijalankan *for loop* atas kategori yang diurutkan, dan pada setiap kategori dicetak lima kata teratas dari sorted() berargumen reverse=True yang dipotong melalui *slicing* [:5].

Modul sqlite3 diimpor, konstanta SIAP ditetapkan sebagai Path("siap_pakai"), dan foldernya dibuat dengan mkdir(exist_ok=True). Isi gabung diubah menjadi list *tuple* (kategori, kata, jumlah), lalu menjadi DataFrame hasil_df berkolom kategori, kata, dan jumlah yang disimpan sebagai siap_pakai/kata_per_kategori.parquet dengan index=False. Sebagai simulasi sqoop export, koneksi SQLite ke siap_pakai/laporan.db dibuka dengan sqlite3.connect(), hasil_df ditulis ke tabel kata_per_kategori melalui to_sql() berargumen if_exists="replace" dan index=False, lalu dicetak jumlah baris tabel dari SELECT COUNT(\*) serta tiga kata dengan jumlah tertinggi pada kategori health. Koneksi ditutup dengan close().

In [22]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3

SIAP = Path("siap_pakai")
SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


# L. Analisis Hasil

## L-1. Dua Proses tanpa Combiner Lebih Lambat daripada Satu Proses

Berdasarkan tabel H-10, dua proses tanpa combiner memerlukan 1,207 detik, sedangkan satu proses tanpa combiner hanya 0,500 detik, sehingga versi paralel 2,4 kali lebih lambat. Kerja mapper pada satu proses hanya setengah detik, sehingga penghematan terbesar yang mungkin diperoleh dari dua CPU hanya sekitar 0,25 detik. Penghematan itu kalah oleh biaya tambahan yang tidak ada pada satu proses. Pada Pool(2), dua proses pekerja harus dibuat terlebih dahulu, setiap proses membaca berkas Parquet sendiri, lalu seluruh hasil mapper diserialisasi dengan pickle, dikirim melalui *pipe* antarproses, dan dibangun ulang di proses utama.

Ketika combiner dipasang, pasangan yang dikirim turun dari 825.601 menjadi 74.009, yaitu 11,2 kali lebih sedikit, dan hasil akhirnya tetap sama persis. Namun pada eksekusi ini, waktu dua proses hampir tidak berubah, yaitu dari 1,207 detik menjadi 1,186 detik atau hanya 0,021 detik lebih cepat. Artinya, pada eksekusi ini sebagian besar biaya dua proses berasal dari pembuatan proses pekerja, bukan dari pengiriman pasangan, sehingga dua proses dengan combiner tetap 2,2 kali lebih lambat daripada satu proses dengan combiner yang memerlukan 0,547 detik. Combiner juga menambah kerja di sisi mapper, terlihat dari satu proses dengan combiner yang 9% lebih lambat daripada tanpa combiner (0,547 berbanding 0,500 detik), karena Counter.update() lebih mahal daripada pembentukan list biasa. Pada *cluster*, pasangan dikirim melalui jaringan sehingga pengurangan 11,2 kali tersebut berdampak jauh lebih besar daripada pengiriman antarproses di satu mesin. Dengan demikian, paralelisme baru menguntungkan apabila kerja setiap mapper jauh lebih besar daripada biaya tetap pembuatan proses dan biaya pemindahan hasilnya, dan kedua syarat itu tidak terpenuhi pada data berukuran 90.818 judul ini.

## L-2. Menambah Reducer Tidak Mendekatkan Beban Terberat ke Bagian yang Adil

Berdasarkan keluaran H-11, beban reducer terberat tanpa combiner adalah 29,17% pada 4 reducer, 8,82% pada 16 reducer, dan 4,28% pada 64 reducer, sedangkan bagian yang adil masing-masing 25,00%, 6,25%, dan 1,56%. Jarak terhadap bagian adil justru melebar, yaitu 1,17 kali pada 4 reducer, 1,41 kali pada 16 reducer, dan 2,74 kali pada 64 reducer. Penyebabnya, partitioner mengirim seluruh pasangan satu kata ke reducer yang sama, dan kata di sendirian sudah menyumbang 27.390 dari 825.601 pasangan, yaitu 3,3%. Berapa pun jumlah reducer, reducer yang menerima kata di menanggung paling sedikit 3,3% beban, sehingga beban terberat tidak dapat turun di bawah angka itu walaupun bagian adilnya terus mengecil. Pada 4 reducer, R0 juga menerima kata ini dengan 10.124 pasangan, sehingga dua kata tersebut saja mengisi 37.514 dari 240.836 pasangan pada R0, atau 15,6%.

Combiner mengubah gambaran ini karena setiap mapper hanya mengirim satu pasangan untuk setiap kata unik, sehingga satu kata menyumbang paling banyak sembilan pasangan, yaitu satu dari setiap split. Beban reducer tidak lagi ditentukan oleh frekuensi kata, melainkan oleh banyaknya kata unik yang jatuh ke reducer tersebut, dan *hash* CRC32 membagi kata unik secara hampir merata. Hasilnya, beban empat reducer menjadi 24,7% sampai 25,6% dengan selisih reducer terberat dan teringan hanya 3,5% (18.911 berbanding 18.267 pasangan), sedangkan tanpa combiner selisihnya 41,6% (240.836 berbanding 170.066 pasangan). Dengan demikian, *skew* yang berasal dari satu kunci tidak diselesaikan dengan menambah reducer, melainkan dengan mengurangi jumlah pasangan per kunci sebelum shuffle.

## L-3. Kata rp, ri, dan the dalam Daftar Stopword

Ketiga kata tersebut tidak perlu diperlakukan sama, karena keputusannya bergantung pada tujuan analisis, yaitu mengetahui topik setiap kategori dan perbedaannya antar-kategori. Kata the muncul 528 kali sebagai kata kelima pada kategori hot dan diduga berasal dari judul atau nama karya berbahasa Inggris. Kata ini adalah kata fungsi bahasa Inggris yang tidak membawa topik, sehingga layak ditambahkan ke STOPWORD. Kata ri muncul 1.125 kali pada finance dan merupakan singkatan Republik Indonesia, sehingga maknanya sama dengan kata indonesia yang sudah masuk lima kata teratas pada empat kategori lain. Kata ini lebih tepat dinormalisasi menjadi indonesia daripada dibuang, karena pembuangan akan mengecilkan frekuensi topik Indonesia pada finance. Apabila tujuannya membuang nama negara yang tidak membedakan kategori, keduanya dibuang bersama.

Kata rp muncul 3.154 kali secara keseluruhan, dengan 1.385 di antaranya pada finance atau 43,9%, dan 267 pada oto. Kata ini menandai judul yang membahas harga atau nilai uang, sehingga berguna untuk membedakan finance dari kategori lain dan sebaiknya dipertahankan apabila tujuannya membandingkan kategori. Sebaliknya, apabila tujuannya mengetahui topik di dalam finance, rp tidak menjelaskan hal yang dibahas sehingga layak dibuang. Karena itu, daftar stopword domain merupakan keputusan analisis yang harus dicatat bersama tujuannya, dan dampaknya terhadap lima kata teratas diuji pada Latihan 1.

## L-4. URL Rusak pada Kategori inet sebagai Kunci Join

Seluruh 5.640 baris inet pada data mentah gagal pada aturan url artikel valid, dan setelah pembersihan tersisa 5.577 baris inet yang kolom url dan id-nya kosong, yaitu 6,1% dari 90.818 baris. Apabila url dipakai sebagai kunci join, akibatnya bergantung pada alat yang dipakai. Pada pandas, merge() memperlakukan nilai kosong pada kunci sebagai nilai yang sama, sehingga setiap baris inet dipasangkan dengan setiap baris berkunci kosong pada tabel lain. Join tabel ini dengan dirinya sendiri, misalnya, menghasilkan 5.577 × 5.577 = 31.102.929 baris palsu dari satu kategori, sedangkan argumen validate="one_to_one" akan menghentikan proses dengan MergeError. Pada SQL, termasuk DuckDB, NULL tidak pernah sama dengan NULL, sehingga inner join membuang seluruh baris inet tanpa pesan galat dan left join mempertahankannya dengan kolom gabungan yang kosong. Kedua akibat itu sama-sama menghasilkan angka yang keliru tanpa tanda pada keluaran.

Baris tersebut tidak dibuang karena cacatnya hanya pada satu kolom, sedangkan judul, kategori, dan tanggalnya lulus seluruh aturan kualitas yang lain. Analisis praktikum ini berbasis judul, dan pembuangan akan menghilangkan satu kategori utuh dari perbandingan antar-kategori, padahal H-13 menunjukkan inet memiliki kata khas seperti ponsel sebanyak 309 kali. Pengosongan url melalui where() menjadi jalan tengah, yaitu judulnya tetap dipakai, sedangkan url yang rusak tidak dapat terpakai sebagai kunci. Pada *pipeline* lanjutan, kunci join sebaiknya memakai kolom yang terisi pada seluruh baris, atau baris berkunci kosong disaring secara eksplisit sebelum join dan jumlahnya dicatat. Cara penanganan id kosong ini dipakai kembali pada Tugas Mandiri.

## L-5. Partisi Kategori Dibanding Partisi Kategori dan Bulan

Partisi kategori menghasilkan 9 berkas dengan total 8,4 MB, sedangkan partisi kategori dan bulan menghasilkan 54 berkas dengan total yang sama, yaitu rata-rata hanya sekitar 0,16 MB per berkas. Pada MapReduce, satu input split dikerjakan oleh satu mapper, sehingga pilihan partisi langsung menentukan jumlah mapper, yaitu 9 mapper berbanding 54 mapper. Setiap mapper memiliki biaya tetap yang tidak bergantung pada ukuran datanya, yaitu penjadwalan *task*, pembukaan berkas, pembacaan metadata Parquet, dan pengiriman hasilnya, sehingga 54 mapper berarti enam kali biaya tetap untuk data yang sama. Keadaan inilah yang disebut *small file problem*. Pada Hadoop, ukuran blok HDFS bawaan adalah 128 MB, sehingga seluruh 8,4 MB data ini bahkan belum mengisi satu blok, sementara setiap berkas tetap dicatat sebagai objek terpisah di memori NameNode dan setiap split memerlukan satu *container* mapper.

Selain itu, pertanyaan praktikum menyangkut kategori, sehingga partisi kategori sudah cukup untuk *partition pruning*, yaitu 1 dari 9 berkas dibaca untuk finance, dan penambahan bulan tidak memberi keuntungan bagi pertanyaan tersebut. Konsekuensi yang diterima adalah split yang tidak seimbang, karena news berisi 32.321 judul sedangkan sport hanya 2.433 judul, yaitu 13,3 kali lipat. Waktu *job* ditentukan oleh mapper paling lambat, sehingga pada data yang jauh lebih besar, pemecahan berkas news menjadi beberapa split dapat dipertimbangkan selama ukuran setiap berkas tetap jauh di atas ukuran yang menimbulkan *small file problem*.

## L-6. Hasil Harus Diurutkan sebelum Masuk ke reducer.py

Hasil harus diurutkan karena reducer.py hanya membandingkan kunci baris saat ini dengan kunci_sekarang, yaitu kunci baris sebelumnya, lalu mencetak total setiap kali kunci berganti. Cara ini hanya benar apabila semua baris berkunci sama datang berurutan tanpa diselingi kunci lain, dan jaminan itu diberikan oleh sort pada *pipeline* lokal H-12 serta oleh tahap shuffle and sort pada Hadoop. Reducer sengaja tidak menyimpan seluruh kunci di memori sehingga dapat memproses masukan yang lebih besar daripada memori.

Apabila sort dihilangkan, keluaran mapper mengikuti urutan kata pada judul, sehingga kata di yang muncul pada ribuan judul selalu diselingi kata lain. Reducer akan mencetak kata yang sama berkali-kali dengan total parsial yang kecil, sehingga keluaran tidak lagi berisi 32.275 baris dengan satu total per kata, melainkan jauh lebih banyak baris yang jumlahnya mendekati 825.601 pasangan dari mapper. Jumlah seluruh nilai tetap 825.601 karena tidak ada pasangan yang hilang, tetapi total per kata menjadi salah, misalnya total 27.390 untuk kata di terpecah ke banyak baris. Kesalahan seperti ini tidak memunculkan galat, sehingga hanya dapat dikenali melalui cek silang seperti perbandingan hasil_pipe dengan hasil_semua pada H-12.

## L-7. Sitemap Berita Dibanding Membuka Setiap Halaman Artikel

Pada H-4, judul diperoleh dari tiga *request* sitemap ditambah satu *request* robots.txt untuk setiap *host*, dengan batas 7 judul per sitemap dan 20 judul total. Membuka halaman artikel satu per satu memerlukan 20 *request* untuk 20 judul yang sama, sehingga dengan DELAY 2,0 detik jeda totalnya menjadi 40 detik, berbanding 6 detik pada sitemap. Satu sitemap berita bahkan memuat 200 entri menurut uji dosen, sehingga satu *request* dapat mewakili hingga 200 halaman artikel. Bagi server detik, cara ini menekan jumlah *request* dan ukuran data yang dikirim, karena sitemap hanya berisi XML ringkas tanpa gambar, iklan, dan skrip yang menyertai halaman artikel. Bagi kode, judul, tanggal terbit, dan kata kunci tersedia pada elemen berstandar protokol sitemaps.org dan sitemap berita, sehingga parser tidak ikut rusak ketika tampilan HTML situs berubah dan dapat diuji tanpa internet seperti pada Langkah 2.

Kerugiannya, kualitas data sepenuhnya bergantung pada ketelitian penyedia sitemap, dan hal ini terlihat langsung pada keluaran H-4. Keluaran H-4 berisi 7 judul news, 6 judul sport, dan 7 judul berkategori kosong, disertai 6 URL duplikat, karena sitemap inet.detik.com/news memuat elemen loc yang hanya berisi CDATA kosong, sebagaimana terlihat pada pemeriksaan berkas sitemap tersebut tanggal 7 November 2026. Akibatnya, url seluruh entri inet terbaca kosong, kategorinya yang diturunkan dari subdomain url juga kosong, dan ketujuh entri tersebut terhitung sebagai satu url yang sama sehingga enam di antaranya tercatat sebagai duplikat. Parser tetap berjalan tanpa galat, sehingga cacat ini hanya terdeteksi melalui pemeriksaan jumlah kosong dan duplikat pada Langkah 4. Selain itu, sitemap berita hanya memuat entri terbaru sehingga tidak dapat dipakai untuk mengambil arsip lama seperti periode Januari sampai Juni 2020 pada dataset, hanya berisi judul dan metadata tanpa isi berita, dan tanda zona waktunya diragukan menurut catatan modul sehingga kode hanya mengambil sepuluh karakter tanggal.

# M. Studi Kasus

## Kasus: Topik Portal Berita pada Paruh Pertama 2020

Pimpinan redaksi ingin mengetahui hal yang dibahas portal berita pada paruh pertama 2020 dan meminta ringkasan satu halaman berdasarkan data judul, dengan memakai temuan dari H-7 dan H-13. Ringkasan berisi 150 sampai 200 kata yang memuat tiga hal, yaitu (a) waktu topik corona memuncak, (b) kategori yang paling banyak membahasnya dan kategori yang bahasannya paling berbeda, serta (c) satu keterbatasan data yang perlu disampaikan kepada pimpinan, misalnya data yang hanya berisi judul atau periode Juni yang tidak penuh.

Kasus dijawab dalam empat bagian. M-1 menghitung judul bertopik corona per bulan, M-2 membandingkan jumlah dan proporsi judul bertopik corona per kategori, M-3 mengukur kemiripan kata teratas antar-kategori untuk menemukan kategori yang bahasannya paling berbeda, dan M-4 berisi ringkasan untuk pimpinan redaksi.

## Deklarasi Konstanta dan Fungsi Studi Kasus

Dilakukan impor terhadap modul collections dan pustaka pandas dengan alias pd. Konstanta KATA_TOPIK bernilai corona sesuai H-7 dan JUMLAH_KATA_PEMBANDING bernilai 20 sebagai banyaknya kata teratas per kategori yang dibandingkan pada M-3, sedangkan *dictionary* PARAMETER_KUERI memetakan KATA_TOPIK ke parameter bernama kata_topik. Kueri KUERI_TOPIK_PER_BULAN dijalankan atas *view* berita dari H-7 dan menghitung per bulan jumlah judul melalui COUNT(\*) serta jumlah judul yang memuat kata topik melalui COUNT(\*) FILTER dengan syarat contains(lower(judul), $kata_topik). KUERI_TOPIK_PER_KATEGORI menghitung jumlah yang sama per kategori dan mengurutkan hasilnya menurun berdasarkan judul_topik.

Fungsi jalankan_kueri_topik(koneksi, kueri) menjalankan kueri dengan PARAMETER_KUERI melalui execute() lalu me-*return* DataFrame dari df(), sedangkan tambah_persen_topik(tabel_topik) menyalin tabel dengan copy() lalu menambahkan kolom persen_judul_topik dua desimal. Fungsi kumpulkan_kata_teratas(penghitung_kategori_kata, jumlah_kata) memindahkan Counter berkunci (kategori, kata) dari H-13 ke defaultdict berisi Counter per kategori, lalu me-*return* *dictionary* kategori ke *set* kata teratas dari most_common(). Fungsi hitung_jaccard(himpunan_pertama, himpunan_kedua) me-*return* indeks Jaccard, yaitu ukuran irisan dibagi ukuran gabungan, dan me-*return* 0,0 apabila gabungannya kosong. Fungsi hitung_kemiripan_kategori(kata_teratas) menjalankan *for loop* atas setiap kategori, menghitung rata-rata indeks Jaccard terhadap kategori lain, dan membentuk kata khas dari selisih *set* kata kategori tersebut terhadap gabungan kata kategori lain. Fungsi tersebut me-*return* DataFrame yang diurutkan menaik melalui sort_values() atas rata_jaccard yang belum dibulatkan, lalu atas kategori apabila nilainya sama persis, sehingga kategori paling berbeda berada di baris pertama dan pembulatan tidak memengaruhi urutan.

Keputusan pada sel ini ada dua. Pertama, topik diukur dari judul yang memuat *string* corona, sama dengan H-7, sehingga kata turunan seperti coronavirus ikut terhitung. Kedua, perbedaan bahasan diukur dengan indeks Jaccard atas 20 kata teratas setelah STOPWORD dibuang, karena ukuran ini membandingkan isi daftar kata tanpa dipengaruhi jumlah judul setiap kategori, dengan konsekuensi urutan dan frekuensi kata di dalam daftar tidak ikut diperhitungkan.

In [23]:
import collections

import pandas as pd

KATA_TOPIK = "corona"
JUMLAH_KATA_PEMBANDING = 20
PARAMETER_KUERI = {"kata_topik": KATA_TOPIK}

KUERI_TOPIK_PER_BULAN = """
    SELECT bulan,
           COUNT(*) AS jumlah_judul,
           COUNT(*) FILTER (WHERE contains(lower(judul), $kata_topik))
               AS judul_topik
    FROM berita
    GROUP BY bulan
    ORDER BY bulan
"""
KUERI_TOPIK_PER_KATEGORI = """
    SELECT kategori,
           COUNT(*) AS jumlah_judul,
           COUNT(*) FILTER (WHERE contains(lower(judul), $kata_topik))
               AS judul_topik
    FROM berita
    GROUP BY kategori
    ORDER BY judul_topik DESC
"""


def jalankan_kueri_topik(koneksi, kueri):
    """Jalankan kueri topik berparameter dan kembalikan hasilnya."""
    return koneksi.execute(kueri, PARAMETER_KUERI).df()


def tambah_persen_topik(tabel_topik):
    """Tambahkan persentase judul yang memuat kata topik."""
    tabel_baru = tabel_topik.copy()
    tabel_baru["persen_judul_topik"] = (
        100 * tabel_baru["judul_topik"] / tabel_baru["jumlah_judul"]
    ).round(2)
    return tabel_baru


def kumpulkan_kata_teratas(penghitung_kategori_kata, jumlah_kata):
    """Kembalikan dict kategori ke set kata teratas dari Counter H-13."""
    per_kategori = collections.defaultdict(collections.Counter)
    for (kategori, kata), frekuensi in penghitung_kategori_kata.items():
        per_kategori[kategori][kata] = frekuensi
    return {
        kategori: {kata for kata, _ in penghitung.most_common(jumlah_kata)}
        for kategori, penghitung in sorted(per_kategori.items())
    }


def hitung_jaccard(himpunan_pertama, himpunan_kedua):
    """Kembalikan ukuran irisan dibagi ukuran gabungan dua himpunan."""
    gabungan = himpunan_pertama | himpunan_kedua
    if not gabungan:
        return 0.0
    return len(himpunan_pertama & himpunan_kedua) / len(gabungan)


def hitung_kemiripan_kategori(kata_teratas):
    """Susun rata-rata kemiripan Jaccard setiap kategori terhadap lainnya.

    Pengurutan memakai nilai yang belum dibulatkan, sehingga kategori
    dengan rata-rata terendah, yaitu yang paling berbeda, berada di baris
    pertama.
    """
    baris_tabel = []
    for kategori, kata_kategori in kata_teratas.items():
        kata_kategori_lain = [
            kata_lain
            for kategori_lain, kata_lain in kata_teratas.items()
            if kategori_lain != kategori
        ]
        skor_kemiripan = [
            hitung_jaccard(kata_kategori, kata_lain)
            for kata_lain in kata_kategori_lain
        ]
        kata_khas = kata_kategori - set().union(*kata_kategori_lain)
        baris_tabel.append({
            "kategori": kategori,
            "rata_jaccard": sum(skor_kemiripan) / len(skor_kemiripan),
            "jumlah_kata_khas": len(kata_khas),
            "kata_khas": ", ".join(sorted(kata_khas)),
        })
    return (pd.DataFrame(baris_tabel)
            .sort_values(["rata_jaccard", "kategori"])
            .reset_index(drop=True))

## M-1. Waktu Topik Corona Memuncak

Fungsi jalankan_kueri_topik() dipanggil dengan koneksi con dari H-7 dan KUERI_TOPIK_PER_BULAN, lalu hasilnya dilengkapi melalui tambah_persen_topik() dan di-*assign* ke topik_per_bulan yang dicetak dengan to_string(index=False). Baris bulan puncak diambil melalui loc[] atas idxmax() kolom judul_topik, lalu nama bulan dan jumlah judulnya dicetak.

Dugaan: jumlah judul_topik per bulan diperkirakan sama persis dengan keluaran H-7, dengan puncak pada Maret 2020.

In [24]:
topik_per_bulan = tambah_persen_topik(
    jalankan_kueri_topik(con, KUERI_TOPIK_PER_BULAN)
)
print(topik_per_bulan.to_string(index=False))

bulan_puncak = topik_per_bulan.loc[topik_per_bulan["judul_topik"].idxmax()]
print(f"\nPuncak topik {KATA_TOPIK}: {bulan_puncak['bulan']} "
      f"dengan {bulan_puncak['judul_topik']:,} judul")

  bulan  jumlah_judul  judul_topik  persen_judul_topik
2020-01         16258          385                2.37
2020-02         16135         1077                6.67
2020-03         18103         6994               38.63
2020-04         16636         4913               29.53
2020-05         16352         2934               17.94
2020-06          7334          943               12.86

Puncak topik corona: 2020-03 dengan 6,994 judul


## Jawaban M-1

Topik corona memuncak pada Maret 2020 dengan 6.994 judul, atau 38,63% dari 18.103 judul bulan itu. Kenaikannya tajam, dari 385 judul pada Januari (2,37%) dan 1.077 judul pada Februari (6,67%) menjadi 6.994 judul pada Maret, yaitu 6,5 kali lipat dalam satu bulan, lalu menurun menjadi 4.913 judul pada April (29,53%) dan 2.934 judul pada Mei (17,94%). Juni hanya mencakup 13 hari pertama, sehingga 943 judul pada bulan itu tidak sebanding dengan bulan penuh.

## M-2. Kategori yang Paling Banyak Membahas Corona

KUERI_TOPIK_PER_KATEGORI dijalankan melalui jalankan_kueri_topik() pada koneksi con, lalu hasilnya dilengkapi dengan tambah_persen_topik(), di-*assign* ke topik_per_kategori, dan dicetak. Dua baris kemudian diambil melalui loc[] atas idxmax(), yaitu kategori dengan judul_topik terbanyak dan kategori dengan persen_judul_topik tertinggi, lalu keduanya dicetak.

Dugaan: news diperkirakan menjadi kategori dengan judul bertopik corona terbanyak karena memuat judul paling banyak, sedangkan health diperkirakan memiliki proporsi tertinggi karena kata corona menempati urutan pertama dengan 2.590 kemunculan di antara 4.914 judulnya pada H-13.

In [25]:
topik_per_kategori = tambah_persen_topik(
    jalankan_kueri_topik(con, KUERI_TOPIK_PER_KATEGORI)
)
print(topik_per_kategori.to_string(index=False))

kategori_terbanyak = topik_per_kategori.loc[
    topik_per_kategori["judul_topik"].idxmax()
]
kategori_proporsi_tertinggi = topik_per_kategori.loc[
    topik_per_kategori["persen_judul_topik"].idxmax()
]
print(f"\nJumlah terbanyak  : {kategori_terbanyak['kategori']} "
      f"({kategori_terbanyak['judul_topik']:,} judul)")
print(f"Proporsi tertinggi: {kategori_proporsi_tertinggi['kategori']} "
      f"({kategori_proporsi_tertinggi['persen_judul_topik']:.2f}% judul)")

kategori  jumlah_judul  judul_topik  persen_judul_topik
    news         32321         7532               23.30
  health          4914         2591               52.73
 finance         14141         2473               17.49
     hot         16290         1663               10.21
  travel          6453         1025               15.88
    inet          5577          847               15.19
     oto          4375          577               13.19
    food          4314          310                7.19
   sport          2433          228                9.37

Jumlah terbanyak  : news (7,532 judul)
Proporsi tertinggi: health (52.73% judul)


## M-3. Kategori dengan Bahasan Paling Berbeda

Fungsi kumpulkan_kata_teratas() dipanggil dengan Counter gabung dari H-13 dan JUMLAH_KATA_PEMBANDING, lalu hasilnya di-*assign* ke kata_teratas_kategori. *Dictionary* tersebut diteruskan ke hitung_kemiripan_kategori() dan hasilnya di-*assign* ke kemiripan_kategori, lalu dicetak melalui to_string() berargumen float_format empat desimal sehingga selisih kecil antar-kategori tetap terlihat. Kategori pada baris pertama melalui loc[0, "kategori"] dicetak sebagai kategori dengan bahasan paling berbeda.

Dugaan: kategori dengan rata-rata kemiripan terendah diperkirakan berasal dari finance atau sport, karena kedua kategori ini memiliki kosakata khas seperti nama perusahaan, bursa, atlet, dan cabang olahraga yang tidak dipakai kategori lain.

In [26]:
kata_teratas_kategori = kumpulkan_kata_teratas(gabung, JUMLAH_KATA_PEMBANDING)
kemiripan_kategori = hitung_kemiripan_kategori(kata_teratas_kategori)
print(kemiripan_kategori.to_string(index=False, float_format="{:.4f}".format))
print("\nKategori dengan bahasan paling berbeda:",
      kemiripan_kategori.loc[0, "kategori"])

kategori  rata_jaccard  jumlah_kata_khas                                                                                                                          kata_khas
   sport        0.0567                17 all, asia, atlet, bryant, ditunda, gol, juara, kejuaraan, kobe, marquez, masters, mcgregor, motogp, olimpiade, rossi, tyson, video
 finance        0.0571                13                                                 as, buat, bumn, ekonomi, erick, gara, ihsg, kerja, mau, naik, pengusaha, t, thohir
    news        0.0682                 9                                                                             anies, dki, hari, kpk, orang, polisi, psbb, warga, wni
  travel        0.0822                13                                bali, china, gunung, hotel, jepang, kota, pantai, penumpang, pesawat, potret, tempat, turis, wisata
    food        0.0823                12                                          aja, ayam, enak, galeri, kuliner, kulineran, makan, makana

## Jawaban M-2 dan M-3

Berdasarkan tabel M-2, kategori dengan judul bertopik corona terbanyak adalah news dengan 7.532 judul, sedangkan kategori dengan proporsi tertinggi adalah health, yaitu 52,73% dari 4.914 judulnya. Kedua ukuran perlu disampaikan bersama, karena jumlah mutlak mengikuti besarnya kategori, sedangkan proporsi menunjukkan seberapa jauh topik itu menguasai bahasan suatu kategori. Kategori dengan proporsi terendah adalah food, yaitu 7,19%. Berdasarkan tabel M-3, kategori dengan bahasan paling berbeda adalah sport dengan rata-rata indeks Jaccard 0,0567 terhadap kategori lain, dan 17 dari 20 kata teratasnya tidak muncul pada kategori lain, misalnya atlet, motogp, olimpiade, dan rossi. Selisihnya dengan finance pada urutan kedua hanya 0,0004, yaitu 0,0567 berbanding 0,0571, sehingga kedua kategori sama-sama memiliki bahasan yang paling jauh dari kategori lain. Sebaliknya, oto dengan 0,1331 dan inet dengan 0,1273 adalah kategori yang kata teratasnya paling banyak beririsan dengan kategori lain.

## M-4. Ringkasan untuk Pimpinan Redaksi

Sepanjang 1 Januari sampai 13 Juni 2020, portal memuat 90.818 judul berita yang lolos pemeriksaan kualitas, dan topik corona menjadi bahasan yang paling menonjol. Topik ini mulai muncul pada Januari dengan 385 judul, naik menjadi 1.077 judul pada Februari, lalu memuncak pada Maret dengan 6.994 judul, yaitu 38,63% dari seluruh judul bulan itu. Setelah puncak tersebut, jumlahnya menurun menjadi 4.913 judul pada April dan 2.934 judul pada Mei. Sepanjang periode, corona menjadi kata yang paling sering muncul pada tujuh dari sembilan kategori. Secara jumlah, kategori news paling banyak membahas corona dengan 7.532 judul, sedangkan secara proporsi, kategori health paling terpengaruh karena 52,73% judulnya memuat kata tersebut. Kategori yang bahasannya paling berbeda adalah sport, dengan kata khas seperti motogp, olimpiade, dan atlet yang tidak muncul di antara kata teratas kategori lain, disusul finance dengan selisih yang sangat tipis. Hasil ini perlu dibaca dengan satu keterbatasan, yaitu data bulan Juni hanya mencakup 13 hari pertama, sehingga 943 judul corona pada bulan itu belum dapat dibandingkan langsung dengan bulan yang penuh dan belum tentu menunjukkan penurunan pemberitaan.

# N. Latihan

## Deklarasi Konstanta dan Fungsi Latihan

Dilakukan impor terhadap modul collections dan time serta pustaka pandas dengan alias pd dan requests. Latihan 1 memakai *set* STOPWORD_DOMAIN berisi rp, ri, the, foto, dan galeri, STOPWORD_LATIHAN sebagai gabungan STOPWORD dari H-9 dan STOPWORD_DOMAIN melalui operator |, serta JUMLAH_KATA_TERATAS bernilai 5. Latihan 2 memakai JUMLAH_KATA_PER_BULAN bernilai 10 serta BULAN_AWAL_BANDING dan BULAN_AKHIR_BANDING bernilai 2020-01 dan 2020-05, dan Latihan 4 memakai *tuple* DAFTAR_JUMLAH_REDUCER berisi 2, 8, dan 32. Latihan 5 memakai JEDA_LATIHAN_DETIK bernilai 0,05, JUMLAH_HALAMAN_UJI bernilai 50, dan JEDA_SITUS_NYATA_DETIK bernilai 1,0. Latihan 6 memakai list SITEMAP_LATIHAN yang sama dengan SITEMAP_DIPILIH, kecuali alamat sport.detik.com/basket diganti finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml yang tercantum pada indeks sitemap.xml detik.com. Latihan 7 memakai PEMISAH_KATA_KUNCI berupa koma dan JUMLAH_KATA_KUNCI_TERATAS bernilai 10.

Fungsi shuffle_pasangan(hasil_map) mengumpulkan nilai setiap kunci dari semua mapper ke defaultdict(list), dan reduce_jumlah(hasil_shuffle) me-*return* *dictionary* jumlah nilai setiap kunci. Untuk Latihan 1 dan 2, petakan_kategori_kata(jalur_split, stopword) dan petakan_bulan_kata(jalur_split, stopword) membaca satu split lalu me-*return* Counter berkunci (kategori, kata) atau (bulan, kata) untuk kata yang tidak termasuk stopword, sehingga combiner terpasang di dalam mapper seperti H-13. Fungsi kata_teratas_per_kelompok(hasil_reduce, jumlah_kata) mengelompokkan *tuple* (frekuensi, kata) per kelompok dan me-*return* kata teratas melalui sorted() berargumen reverse=True, yaitu urutan yang sama dengan H-13. Fungsi format_daftar_kata(daftar_kata) mengubah list (kata, frekuensi) menjadi teks, bandingkan_kata_teratas(teratas_sebelum, teratas_sesudah) me-*return* DataFrame berisi kata sebelum, kata sesudah, kata yang keluar, dan kata yang masuk per kategori, sedangkan bandingkan_dua_kelompok(teratas, kelompok_awal, kelompok_akhir) me-*return* *dictionary* kata yang hanya ada pada kelompok awal, hanya pada kelompok akhir, dan pada keduanya.

Untuk Latihan 3, bentuk_bigram(judul) me-*return* list pasangan kata berurutan dari zip() atas token dan token yang digeser satu posisi, petakan_bigram(jalur_split) me-*return* list (bigram, 1) tanpa combiner, dan petakan_bigram_dengan_combiner(jalur_split) me-*return* Counter bigram per split. Untuk Latihan 4, ringkas_beban_reducer(pasangan_per_mapper, daftar_jumlah_reducer) memanggil beban() dari H-11 pada setiap jumlah reducer dan me-*return* DataFrame berisi bagian yang adil, persentase reducer terberat, dan rasio keduanya. Untuk Latihan 5, ukur_crawl(daftar_halaman, jeda_detik) menjalankan crawl_iter() dari H-3 dan me-*return* jumlah judul beserta durasinya, sedangkan perkirakan_durasi() menghitung biaya kerja per halaman dari durasi terukur dikurangi total jeda, lalu me-*return* perkiraan durasi untuk jumlah halaman dan jeda target. Untuk Latihan 6, crawl_sitemap(daftar_sitemap) mengulang alur crawl_detik() dari H-4 dengan daftar sitemap sebagai parameter dan sesi requests yang dibuka di dalam blok with, sedangkan ringkas_aturan_kualitas(data) me-*return* tabel jumlah dan persentase gagal setiap aturan dari aturan_kualitas() di H-6. Untuk Latihan 7, mapper_kata_kunci(entri) me-*return* list (kata_kunci, 1) dari kolom kata_kunci yang dipisah koma, dirapikan dengan strip(), dan diubah ke huruf kecil, sedangkan reducer_kata_kunci(kata_kunci, daftar_nilai) me-*return* kata kunci beserta jumlah nilainya.

In [27]:
import collections
import time

import pandas as pd
import requests

STOPWORD_DOMAIN = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_LATIHAN = STOPWORD | STOPWORD_DOMAIN
JUMLAH_KATA_TERATAS = 5

JUMLAH_KATA_PER_BULAN = 10
BULAN_AWAL_BANDING = "2020-01"
BULAN_AKHIR_BANDING = "2020-05"

DAFTAR_JUMLAH_REDUCER = (2, 8, 32)

JEDA_LATIHAN_DETIK = 0.05
JUMLAH_HALAMAN_UJI = 50
JEDA_SITUS_NYATA_DETIK = 1.0

SITEMAP_LATIHAN = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml",
]

PEMISAH_KATA_KUNCI = ","
JUMLAH_KATA_KUNCI_TERATAS = 10


def shuffle_pasangan(hasil_map):
    """Kelompokkan nilai dari semua mapper berdasarkan kuncinya."""
    hasil_shuffle = collections.defaultdict(list)
    for daftar_pasangan in hasil_map:
        for kunci, nilai in daftar_pasangan:
            hasil_shuffle[kunci].append(nilai)
    return hasil_shuffle


def reduce_jumlah(hasil_shuffle):
    """Jumlahkan seluruh nilai setiap kunci sebagai tahap reduce."""
    return {
        kunci: sum(daftar_nilai)
        for kunci, daftar_nilai in hasil_shuffle.items()
    }


def petakan_kategori_kata(jalur_split, stopword):
    """Hitung pasangan (kategori, kata) satu split dengan combiner."""
    kategori = jalur_split.parent.name.split("=")[1]
    daftar_judul = pd.read_parquet(jalur_split, columns=["judul"])["judul"]
    penghitung = collections.Counter()
    for judul in daftar_judul:
        for kata in tokenisasi(judul):
            if kata not in stopword:
                penghitung[(kategori, kata)] += 1
    return penghitung


def petakan_bulan_kata(jalur_split, stopword):
    """Hitung pasangan (bulan, kata) satu split dengan combiner."""
    data_split = pd.read_parquet(jalur_split, columns=["judul", "bulan"])
    penghitung = collections.Counter()
    for judul, bulan in zip(data_split["judul"], data_split["bulan"]):
        for kata in tokenisasi(judul):
            if kata not in stopword:
                penghitung[(bulan, kata)] += 1
    return penghitung


def kata_teratas_per_kelompok(hasil_reduce, jumlah_kata):
    """Kembalikan dict kelompok ke list (kata, frekuensi) teratas.

    Pengurutan memakai (frekuensi, kata) menurun seperti H-13 sehingga
    urutan kata berfrekuensi sama tetap konsisten.
    """
    per_kelompok = collections.defaultdict(list)
    for (kelompok, kata), frekuensi in hasil_reduce.items():
        per_kelompok[kelompok].append((frekuensi, kata))
    return {
        kelompok: [
            (kata, frekuensi)
            for frekuensi, kata in sorted(isi, reverse=True)[:jumlah_kata]
        ]
        for kelompok, isi in sorted(per_kelompok.items())
    }


def format_daftar_kata(daftar_kata):
    """Ubah list (kata, frekuensi) menjadi teks yang mudah dibaca."""
    return ", ".join(
        f"{kata} ({frekuensi:,})" for kata, frekuensi in daftar_kata
    )


def bandingkan_kata_teratas(teratas_sebelum, teratas_sesudah):
    """Susun tabel kata teratas sebelum dan sesudah perubahan stopword."""
    baris_tabel = []
    for kategori, daftar_sebelum in teratas_sebelum.items():
        kata_sebelum = [kata for kata, _ in daftar_sebelum]
        kata_sesudah = [kata for kata, _ in teratas_sesudah[kategori]]
        baris_tabel.append({
            "kategori": kategori,
            "sebelum": format_daftar_kata(daftar_sebelum),
            "sesudah": format_daftar_kata(teratas_sesudah[kategori]),
            "kata_keluar": ", ".join(
                kata for kata in kata_sebelum if kata not in kata_sesudah
            ),
            "kata_masuk": ", ".join(
                kata for kata in kata_sesudah if kata not in kata_sebelum
            ),
        })
    return pd.DataFrame(baris_tabel)


def bandingkan_dua_kelompok(teratas, kelompok_awal, kelompok_akhir):
    """Kembalikan kata yang hilang, muncul, dan bertahan antar-kelompok."""
    kata_awal = [kata for kata, _ in teratas[kelompok_awal]]
    kata_akhir = [kata for kata, _ in teratas[kelompok_akhir]]
    return {
        "hanya_awal": [kata for kata in kata_awal if kata not in kata_akhir],
        "hanya_akhir": [kata for kata in kata_akhir if kata not in kata_awal],
        "bertahan": [kata for kata in kata_awal if kata in kata_akhir],
    }


def bentuk_bigram(judul):
    """Kembalikan list pasangan dua kata berurutan pada satu judul."""
    token = tokenisasi(judul)
    return [
        f"{kata_pertama} {kata_kedua}"
        for kata_pertama, kata_kedua in zip(token, token[1:])
    ]


def petakan_bigram(jalur_split):
    """Hasilkan pasangan (bigram, 1) satu split tanpa combiner."""
    daftar_judul = pd.read_parquet(jalur_split, columns=["judul"])["judul"]
    return [
        (bigram, 1)
        for judul in daftar_judul
        for bigram in bentuk_bigram(judul)
    ]


def petakan_bigram_dengan_combiner(jalur_split):
    """Hitung bigram satu split di sisi mapper sebelum shuffle."""
    daftar_judul = pd.read_parquet(jalur_split, columns=["judul"])["judul"]
    penghitung = collections.Counter()
    for judul in daftar_judul:
        penghitung.update(bentuk_bigram(judul))
    return penghitung


def ringkas_beban_reducer(pasangan_per_mapper, daftar_jumlah_reducer):
    """Bandingkan beban reducer terberat dengan bagian yang adil."""
    baris_tabel = []
    for jumlah_reducer in daftar_jumlah_reducer:
        beban_reducer = beban(pasangan_per_mapper, jumlah_reducer)
        persen_terberat = 100 * max(beban_reducer) / sum(beban_reducer)
        bagian_adil = 100 / jumlah_reducer
        baris_tabel.append({
            "jumlah_reducer": jumlah_reducer,
            "bagian_adil_persen": round(bagian_adil, 2),
            "terberat_persen": round(persen_terberat, 2),
            "rasio_terhadap_adil": round(persen_terberat / bagian_adil, 2),
        })
    return pd.DataFrame(baris_tabel)


def ukur_crawl(daftar_halaman, jeda_detik):
    """Crawl portal lokal, lalu kembalikan jumlah judul dan durasinya."""
    waktu_mulai = time.perf_counter()
    daftar_judul = list(crawl_iter(BASE, daftar_halaman, delay=jeda_detik))
    return len(daftar_judul), time.perf_counter() - waktu_mulai


def perkirakan_durasi(durasi_terukur, jumlah_halaman_uji, jeda_uji,
                      jumlah_halaman_target, jeda_target):
    """Perkirakan durasi crawl dari biaya kerja per halaman ditambah jeda."""
    total_jeda_uji = jumlah_halaman_uji * jeda_uji
    biaya_kerja_per_halaman = (
        (durasi_terukur - total_jeda_uji) / jumlah_halaman_uji
    )
    return jumlah_halaman_target * (biaya_kerja_per_halaman + jeda_target)


def crawl_sitemap(daftar_sitemap):
    """Ambil judul dari daftar sitemap dengan batas dan jeda seperti H-4."""
    hasil = []
    with requests.Session() as sesi:
        sesi.headers["User-Agent"] = UA_DETIK
        for alamat_sitemap in daftar_sitemap:
            if len(hasil) >= BATAS_TOTAL:
                break
            entri_sitemap = parse_sitemap_news(ambil_xml(alamat_sitemap, sesi))
            hasil += entri_sitemap[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]


def ringkas_aturan_kualitas(data):
    """Kembalikan jumlah dan persentase baris gagal setiap aturan H-6."""
    baris_tabel = []
    for nama_aturan, lulus in aturan_kualitas(data).items():
        baris_tabel.append({
            "aturan": nama_aturan,
            "gagal": int((~lulus).sum()),
            "persen_gagal": round(100 * (~lulus).mean(), 2),
        })
    return pd.DataFrame(baris_tabel)


def mapper_kata_kunci(entri):
    """Hasilkan pasangan (kata_kunci, 1) dari satu entri hasil crawl."""
    teks_kata_kunci = entri.get("kata_kunci") or ""
    return [
        (kata_kunci.strip().lower(), 1)
        for kata_kunci in teks_kata_kunci.split(PEMISAH_KATA_KUNCI)
        if kata_kunci.strip()
    ]


def reducer_kata_kunci(kata_kunci, daftar_nilai):
    """Jumlahkan kemunculan satu kata kunci."""
    return kata_kunci, sum(daftar_nilai)

## Latihan 1: Tambahkan 5 kata ke STOPWORD yang mengganggu hasil, lalu jalankan ulang H-13

Mapper petakan_kategori_kata() dijalankan atas setiap jalur pada splits dua kali, yaitu dengan STOPWORD asli ke hasil_map_sebelum dan dengan STOPWORD_LATIHAN ke hasil_map_sesudah. Setiap Counter diubah menjadi pasangan melalui items(), dikumpulkan dengan shuffle_pasangan(), dan dijumlahkan dengan reduce_jumlah(), lalu lima kata teratas per kategori diambil melalui kata_teratas_per_kelompok() dengan JUMLAH_KATA_TERATAS. Perbandingan keduanya disusun dengan bandingkan_kata_teratas() dan dicetak bersama daftar kata yang ditambahkan ke stopword.

Keputusan pada sel ini adalah pemilihan lima kata stopword domain berdasarkan lima kata teratas H-13. Kata the adalah kata fungsi bahasa Inggris, ri adalah singkatan yang maknanya sama dengan indonesia, rp adalah satuan mata uang yang tidak menyebut topik, sedangkan foto dan galeri diduga berasal dari label format artikel foto, bukan dari topik beritanya. Konsekuensinya, informasi bahwa suatu judul membahas nilai uang atau berbentuk galeri foto ikut hilang dari hitungan.

Dugaan: lima kata teratas diperkirakan berubah pada lima kategori, yaitu finance karena rp dan ri keluar, food karena foto dan galeri keluar, serta hot, oto, dan travel karena masing-masing kehilangan the, rp, dan foto. Empat kategori lain, yaitu health, inet, news, dan sport, diperkirakan tidak berubah karena kelima kata tersebut tidak ada pada lima kata teratasnya.

In [28]:
hasil_map_sebelum = [
    petakan_kategori_kata(jalur, STOPWORD) for jalur in splits
]
hasil_map_sesudah = [
    petakan_kategori_kata(jalur, STOPWORD_LATIHAN) for jalur in splits
]
hasil_reduce_sebelum = reduce_jumlah(
    shuffle_pasangan(penghitung.items() for penghitung in hasil_map_sebelum)
)
hasil_reduce_sesudah = reduce_jumlah(
    shuffle_pasangan(penghitung.items() for penghitung in hasil_map_sesudah)
)

teratas_sebelum = kata_teratas_per_kelompok(
    hasil_reduce_sebelum, JUMLAH_KATA_TERATAS
)
teratas_sesudah = kata_teratas_per_kelompok(
    hasil_reduce_sesudah, JUMLAH_KATA_TERATAS
)
perbandingan_stopword = bandingkan_kata_teratas(
    teratas_sebelum, teratas_sesudah
)

print("Kata yang ditambahkan ke stopword:", ", ".join(sorted(STOPWORD_DOMAIN)))
print(perbandingan_stopword.to_string(index=False))

Kata yang ditambahkan ke stopword: foto, galeri, ri, rp, the
kategori                                                                       sebelum                                                                       sesudah  kata_keluar   kata_masuk
 finance             corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)           corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)       rp, ri mau, ekonomi
    food              makanan (699), makan (539), foto (458), galeri (438), enak (358)              makanan (699), makan (539), enak (358), bikin (343), rumah (311) foto, galeri bikin, rumah
  health corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413) corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)                          
     hot           corona (1,658), anak (793), rumah (561), meninggal (545), the (528)          corona (1,658), anak (793), rumah (561), meninggal (545), film (489)          t

## Jawaban Latihan 1

Kata yang ditambahkan ke stopword adalah rp, ri, the, foto, dan galeri. Lima kata teratas berubah pada lima kategori, sesuai dugaan. Pada finance, rp dan ri keluar sehingga mau (562) dan ekonomi (536) masuk, sedangkan pada food, foto dan galeri digantikan oleh bikin (343) dan rumah (311). Pada hot, the digantikan film (489), pada oto, rp digantikan honda (255), dan pada travel, foto digantikan indonesia (269). Kategori health, inet, news, dan sport tidak berubah karena kelima kata tersebut tidak termasuk lima kata teratasnya. Kata yang masuk pada finance, hot, dan oto, yaitu ekonomi, film, dan honda, lebih mencerminkan topik kategorinya. Sebaliknya, kata yang masuk pada food dan finance juga memuat kata umum seperti bikin, rumah, dan mau, sehingga stopword domain masih perlu diperluas apabila tujuannya menampilkan topik saja. Konsekuensinya, informasi tentang judul yang membahas harga atau artikel galeri foto tidak lagi terlihat.

## Latihan 2: Ubah key di H-13 menjadi (bulan, kata), lalu amati kata yang berubah dari Januari ke Mei

Mapper petakan_bulan_kata() dijalankan atas setiap split dengan STOPWORD asli, lalu hasilnya dikumpulkan dengan shuffle_pasangan() dan dijumlahkan dengan reduce_jumlah() ke kata_per_bulan. Sepuluh kata teratas setiap bulan diambil melalui kata_teratas_per_kelompok() dengan JUMLAH_KATA_PER_BULAN, lalu dijalankan *for loop* atas BULAN_AWAL_BANDING dan BULAN_AKHIR_BANDING untuk mencetak kata teratas kedua bulan tersebut. Kata yang hanya ada pada Januari, hanya pada Mei, dan pada keduanya dihitung dengan bandingkan_dua_kelompok() lalu dicetak.

Keputusan pada sel ini adalah pemakaian STOPWORD asli, bukan STOPWORD_LATIHAN, sehingga hasilnya dapat dibandingkan langsung dengan H-13 dan hanya berbeda pada kuncinya.

Dugaan: kata teratas Januari diperkirakan didominasi peristiwa awal tahun seperti banjir, sedangkan Mei diperkirakan didominasi corona beserta kata terkait pembatasan sosial dan hari raya, seperti psbb dan lebaran.

In [29]:
hasil_map_bulan = [petakan_bulan_kata(jalur, STOPWORD) for jalur in splits]
kata_per_bulan = reduce_jumlah(
    shuffle_pasangan(penghitung.items() for penghitung in hasil_map_bulan)
)
teratas_per_bulan = kata_teratas_per_kelompok(
    kata_per_bulan, JUMLAH_KATA_PER_BULAN
)
for bulan in (BULAN_AWAL_BANDING, BULAN_AKHIR_BANDING):
    print(f"{bulan}: {format_daftar_kata(teratas_per_bulan[bulan])}")

perubahan_kata = bandingkan_dua_kelompok(
    teratas_per_bulan, BULAN_AWAL_BANDING, BULAN_AKHIR_BANDING
)
print(f"\nHanya pada {BULAN_AWAL_BANDING}:",
      ", ".join(perubahan_kata["hanya_awal"]))
print(f"Hanya pada {BULAN_AKHIR_BANDING}:",
      ", ".join(perubahan_kata["hanya_akhir"]))
print("Bertahan pada keduanya:", ", ".join(perubahan_kata["bertahan"]))

2020-01: banjir (1,220), rp (616), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428), minta (427), kpk (415), indonesia (410)
2020-05: corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562), rp (558), psbb (522), covid (522), polisi (517)

Hanya pada 2020-01: banjir, jokowi, tahun, baru, mobil, minta, kpk, indonesia
Hanya pada 2020-05: corona, kasus, positif, pandemi, warga, lebaran, psbb, covid
Bertahan pada keduanya: rp, polisi


## Jawaban Latihan 2

Dari Januari ke Mei, delapan dari sepuluh kata teratas berganti. Kata yang keluar adalah banjir, jokowi, tahun, baru, mobil, minta, kpk, dan indonesia, sedangkan kata yang masuk adalah corona, kasus, positif, pandemi, warga, lebaran, psbb, dan covid. Hanya rp dan polisi yang bertahan pada kedua bulan. Pada Januari, kata teratas adalah banjir dengan 1.220 kemunculan dan corona belum masuk sepuluh besar, sedangkan pada Mei corona menempati urutan pertama dengan 2.935 kemunculan. Perubahan ini menunjukkan pergeseran bahasan dari peristiwa awal tahun, terutama banjir, ke pandemi beserta dampaknya, yaitu kasus, PSBB, dan lebaran, sejalan dengan puncak topik corona pada Maret yang terlihat di H-7.

## Latihan 3: Ubah mapper agar menghasilkan bigram, lalu hitung pasangan sebelum dan sesudah combiner

Mapper petakan_bigram() dijalankan atas setiap split ke hasil_map_bigram, sedangkan petakan_bigram_dengan_combiner() dijalankan ke hasil_map_bigram_combiner. Jumlah pasangan yang dikirim dihitung dari penjumlahan panjang setiap list atau Counter, lalu dicetak jumlah pasangan tanpa combiner, jumlah pasangan dengan combiner, dan rasio keduanya.

Keputusan pada sel ini adalah pembentukan bigram dari seluruh token tanpa membuang STOPWORD, sehingga pasangan seperti virus corona tetap utuh dan jumlah pasangan dapat dibandingkan langsung dengan 825.601 pasangan kata tunggal pada H-9.

Dugaan: jumlah bigram tanpa combiner diperkirakan sekitar 734.783, yaitu 825.601 token dikurangi 90.818 judul, karena judul dengan n token menghasilkan n − 1 bigram. Penghematan combiner diperkirakan jauh lebih kecil daripada 11,2 kali pada H-10, karena sebagian besar bigram hanya muncul satu kali di dalam satu split.

In [30]:
hasil_map_bigram = [petakan_bigram(jalur) for jalur in splits]
hasil_map_bigram_combiner = [
    petakan_bigram_dengan_combiner(jalur) for jalur in splits
]
pasangan_tanpa_combiner = sum(len(daftar) for daftar in hasil_map_bigram)
pasangan_dengan_combiner = sum(
    len(penghitung) for penghitung in hasil_map_bigram_combiner
)

print(f"Pasangan tanpa combiner : {pasangan_tanpa_combiner:,}")
print(f"Pasangan dengan combiner: {pasangan_dengan_combiner:,}")
print(f"Pengurangan             : "
      f"{pasangan_tanpa_combiner / pasangan_dengan_combiner:.2f} kali")

Pasangan tanpa combiner : 734,783
Pasangan dengan combiner: 411,217
Pengurangan             : 1.79 kali


## Jawaban Latihan 3

Mapper bigram menghasilkan 734.783 pasangan tanpa combiner, tepat sama dengan 825.601 token dikurangi 90.818 judul, dan 411.217 pasangan dengan combiner. Combiner hanya mengurangi pasangan 1,79 kali, jauh di bawah 11,2 kali pada word count H-10. Penyebabnya, rata-rata satu bigram hanya muncul 1,79 kali di dalam satu split, sedangkan satu kata tunggal rata-rata muncul 11,2 kali, sehingga tidak banyak pasangan yang dapat digabungkan oleh combiner. Semakin panjang kunci, semakin beragam kombinasinya dan semakin kecil manfaat combiner, sehingga biaya shuffle untuk bigram tetap besar walaupun combiner dipasang.

## Latihan 4: Jalankan H-11 dengan R = 2, 8, dan 32, lalu buat tabel beban reducer terberat dibanding bagian yang adil

Fungsi ringkas_beban_reducer() dipanggil dengan hasil_map dari H-9 tanpa combiner dan DAFTAR_JUMLAH_REDUCER, sama seperti perbandingan bagian adil pada H-11, lalu hasilnya di-*assign* ke tabel_beban_reducer dan dicetak.

Dugaan: rasio beban terberat terhadap bagian adil diperkirakan naik seiring jumlah reducer, dan pada 32 reducer bagian adilnya 3,125% sehingga sudah di bawah 3,3% milik kata di, artinya reducer terberat pasti melampaui bagian adil.

In [31]:
tabel_beban_reducer = ringkas_beban_reducer(hasil_map, DAFTAR_JUMLAH_REDUCER)
print(tabel_beban_reducer.to_string(index=False))

 jumlah_reducer  bagian_adil_persen  terberat_persen  rasio_terhadap_adil
              2               50.00            52.27                 1.05
              8               12.50            16.83                 1.35
             32                3.12             5.82                 1.86


## Jawaban Latihan 4

| Jumlah reducer | Bagian adil | Beban reducer terberat | Rasio terhadap bagian adil |
|---|---|---|---|
| 2 | 50,00% | 52,27% | 1,05 |
| 8 | 12,50% | 16,83% | 1,35 |
| 32 | 3,12% | 5,82% | 1,86 |

Berdasarkan tabel di atas, rasio beban terberat terhadap bagian adil naik dari 1,05 kali pada 2 reducer menjadi 1,35 kali pada 8 reducer dan 1,86 kali pada 32 reducer, sehingga pola H-11 terulang, yaitu makin banyak reducer, makin jauh reducer terberat dari bagian adilnya. Pada 32 reducer, bagian adil sudah lebih kecil daripada 3,3% beban kata di, sehingga reducer yang memegang kata itu pasti melampaui bagian adil berapa pun pembagian kata lainnya. Penambahan reducer tetap menurunkan beban terberat dalam persen, yaitu dari 52,27% menjadi 5,82%, tetapi tidak dapat menghilangkan *skew* yang berasal dari kata yang sangat sering muncul.

## Latihan 5: Tambahkan jeda 0,05 detik pada crawl_iter untuk 50 halaman pertama, lalu perkirakan waktu 911 halaman dengan jeda 1 detik

Fungsi ukur_crawl() dijalankan atas range(1, JUMLAH_HALAMAN_UJI + 1) dengan JEDA_LATIHAN_DETIK, lalu jumlah judul dan durasinya di-*assign* ke jumlah_judul_uji dan durasi_uji. Perkiraan durasi untuk N_HALAMAN halaman dengan JEDA_SITUS_NYATA_DETIK dihitung melalui perkirakan_durasi() dan di-*assign* ke perkiraan_durasi_penuh. Dicetak jumlah judul dan durasi uji, serta perkiraan durasi penuh dalam detik dan menit.

Keputusan pada sel ini adalah cara perkiraan, yaitu biaya kerja per halaman diukur dari 50 halaman uji setelah total jeda dikurangkan, lalu dikalikan 911 halaman dan ditambah jeda 1 detik per halaman. Cara ini mengasumsikan biaya kerja setiap halaman sama dan server lokal sama cepatnya dengan server sungguhan, sehingga hasilnya merupakan batas bawah untuk situs nyata yang lebih lambat.

Dugaan: 50 halaman dengan jeda 0,05 detik diperkirakan selesai dalam beberapa detik karena total jedanya hanya 2,5 detik, sedangkan 911 halaman dengan jeda 1 detik diperkirakan memerlukan lebih dari 911 detik, yaitu sekitar 15 menit ditambah biaya kerjanya.

In [32]:
jumlah_judul_uji, durasi_uji = ukur_crawl(
    range(1, JUMLAH_HALAMAN_UJI + 1), JEDA_LATIHAN_DETIK
)
perkiraan_durasi_penuh = perkirakan_durasi(
    durasi_uji, JUMLAH_HALAMAN_UJI, JEDA_LATIHAN_DETIK,
    N_HALAMAN, JEDA_SITUS_NYATA_DETIK,
)

print(f"Judul dari {JUMLAH_HALAMAN_UJI} halaman: {jumlah_judul_uji:,}")
print(f"Durasi uji dengan jeda {JEDA_LATIHAN_DETIK} detik: "
      f"{durasi_uji:.2f} detik")
print(f"Perkiraan {N_HALAMAN} halaman, jeda {JEDA_SITUS_NYATA_DETIK} detik: "
      f"{perkiraan_durasi_penuh:,.1f} detik "
      f"({perkiraan_durasi_penuh / 60:.1f} menit)")

Judul dari 50 halaman: 5,000
Durasi uji dengan jeda 0.05 detik: 5.52 detik
Perkiraan 911 halaman, jeda 1.0 detik: 966.0 detik (16.1 menit)


## Jawaban Latihan 5

Crawl 50 halaman dengan jeda 0,05 detik menghasilkan 5.000 judul dalam 5,52 detik, dengan 2,5 detik di antaranya berupa jeda. Dengan jeda 1 detik, 911 halaman diperkirakan memerlukan 966,0 detik atau 16,1 menit, dan 911 detik di antaranya hanya jeda. Waktu crawl 911 halaman tanpa jeda pada H-5 adalah 51,6 detik, sehingga jeda 1 detik membuat crawling sekitar 18,7 kali lebih lama. Pada situs sungguhan, biaya ini sengaja diterima karena jeda melindungi server dari lonjakan *request*, sehingga data berskala besar sebaiknya diambil dari sumber lain, seperti sitemap atau unduhan resmi, bukan dari ribuan halaman.

## Latihan 6: Ganti satu alamat di SITEMAP_DIPILIH dengan sitemap_news dari bagian lain, lalu terapkan aturan_kualitas dari H-6 pada 20 judul hasil crawl

Fungsi crawl_sitemap() dijalankan atas SITEMAP_LATIHAN di dalam blok *try-except*, dan *exception* OSError, yang mencakup galat requests dan PermissionError, serta RuntimeError dari status 403 atau 429 ditangani dengan meng-*assign* list kosong ke sampel_latihan dan mencetak penanda [GAGAL] beserta pesannya. *If statement* berikutnya hanya berjalan apabila sampel_latihan berisi data. Entri diubah menjadi DataFrame data_latihan, lalu dicetak penanda [OK] beserta jumlah judul, sebaran kategori melalui value_counts().to_dict(), dan tabel hasil ringkas_aturan_kualitas().

Keputusan pada sel ini adalah pemilihan sitemap finance.detik.com/berita-ekonomi-bisnis sebagai pengganti sport.detik.com/basket, karena kategori finance tersedia pada dataset sehingga aturan kategori dikenal dapat diuji pada kategori yang sama dengan data latih. Batas BATAS_PER_BAGIAN, BATAS_TOTAL, dan DELAY dari H-4 tetap dipakai, sehingga sel ini hanya mengirim tiga *request* sitemap dan sebaiknya tidak dijalankan berulang.

Dugaan: kategori yang muncul diperkirakan news dan finance, sedangkan entri dari sitemap inet diperkirakan kembali berkategori kosong karena elemen loc pada sitemap tersebut hanya berisi CDATA kosong, sehingga aturan kategori dikenal dan url artikel valid diperkirakan gagal pada entri inet.

In [33]:
try:
    sampel_latihan = crawl_sitemap(SITEMAP_LATIHAN)
except (OSError, RuntimeError) as galat:
    sampel_latihan = []
    print("[GAGAL] Crawling:", type(galat).__name__, "-", galat)

if sampel_latihan:
    data_latihan = pd.DataFrame(sampel_latihan)
    print("[OK] Judul terkumpul:", len(data_latihan))
    print("Kategori:", data_latihan["kategori"].value_counts().to_dict())
    print(ringkas_aturan_kualitas(data_latihan).to_string(index=False))

[OK] Judul terkumpul: 20
Kategori: {'news': 7, '': 7, 'finance': 6}
              aturan  gagal  persen_gagal
  judul tidak kosong      0           0.0
     judul 3-30 kata      0           0.0
    kategori dikenal      7          35.0
       tanggal valid      0           0.0
   url artikel valid      7          35.0
judul tidak duplikat      0           0.0


## Jawaban Latihan 6

Setelah alamat sport.detik.com/basket diganti finance.detik.com/berita-ekonomi-bisnis, kategori yang muncul adalah news sebanyak 7 judul, finance sebanyak 6 judul, dan 7 judul berkategori kosong yang berasal dari sitemap inet. Kategori finance diperoleh dari subdomain URL, sama seperti kolom kategori pada CSV. Penerapan aturan_kualitas pada 20 judul hasil crawl menunjukkan bahwa aturan judul tidak kosong, judul 3-30 kata, tanggal valid, dan judul tidak duplikat tidak gagal pada satu judul pun, sedangkan aturan kategori dikenal dan url artikel valid sama-sama gagal pada 7 judul atau 35%. Kegagalan kedua aturan tersebut berasal dari entri sitemap inet.detik.com/news yang elemen loc-nya hanya berisi CDATA kosong, sehingga url-nya kosong dan kategori yang diturunkan dari subdomain url juga kosong. Hasil ini menunjukkan bahwa aturan kualitas H-6 yang dirancang untuk dataset 2020 tetap berguna untuk data hasil crawl baru, karena cacat pada sumber terdeteksi sebelum data masuk ke lapisan berikutnya.

## Latihan 7: Tulis mapper dan reducer untuk menghitung kata kunci terbanyak pada hasil crawl

Entri hasil crawl H-4 pada sampel dan hasil crawl Latihan 6 pada sampel_latihan digabung ke semua_entri. Entri dengan judul yang sama dibuang melalui *dictionary comprehension* berkunci judul, lalu nilai *dictionary* tersebut di-*assign* ke entri_unik. Tahap map menjalankan mapper_kata_kunci() atas setiap entri, tahap shuffle memakai shuffle_pasangan(), dan tahap reduce menjalankan reducer_kata_kunci() atas setiap kata kunci. Hasilnya diurutkan dengan sorted() berkunci *lambda* atas jumlah menurun dan kata kunci menaik, lalu dicetak jumlah entri, jumlah kata kunci unik, dan JUMLAH_KATA_KUNCI_TERATAS kata kunci teratas.

Keputusan pada sel ini ada dua. Pertama, kata kunci dirapikan dengan strip() dan diubah ke huruf kecil sehingga Corona dan corona dihitung sebagai satu kata kunci, dengan konsekuensi perbedaan penulisan nama diri ikut hilang. Kedua, duplikat antara kedua hasil crawl dibuang berdasarkan judul, bukan url, karena url entri inet kosong sehingga penghapusan berdasarkan url akan menggabungkan seluruh entri inet menjadi satu, dengan risiko dua berita berbeda yang berjudul sama ikut tergabung.

Dugaan: jumlah entri unik diperkirakan sekitar 26, yaitu 7 news dan 7 inet yang sama pada kedua hasil crawl, 6 sport, dan 6 finance, dan jumlahnya bertambah apabila sitemap news diperbarui di antara kedua crawl. Kata kunci teratas diperkirakan hanya muncul dua atau tiga kali karena jumlah entrinya kecil.

In [34]:
semua_entri = sampel + sampel_latihan
entri_unik = list({entri["judul"]: entri for entri in semua_entri}.values())

hasil_map_kata_kunci = [mapper_kata_kunci(entri) for entri in entri_unik]
hasil_shuffle_kata_kunci = shuffle_pasangan(hasil_map_kata_kunci)
frekuensi_kata_kunci = dict(
    reducer_kata_kunci(kata_kunci, daftar_nilai)
    for kata_kunci, daftar_nilai in hasil_shuffle_kata_kunci.items()
)
peringkat_kata_kunci = sorted(
    frekuensi_kata_kunci.items(),
    key=lambda pasangan: (-pasangan[1], pasangan[0]),
)

print(f"Entri hasil crawl : {len(semua_entri):,}")
print(f"Entri unik        : {len(entri_unik):,}")
print(f"Kata kunci unik   : {len(frekuensi_kata_kunci):,}")
print(f"\n{JUMLAH_KATA_KUNCI_TERATAS} kata kunci teratas:")
for kata_kunci, jumlah in peringkat_kata_kunci[:JUMLAH_KATA_KUNCI_TERATAS]:
    print(f"  {kata_kunci:<35} {jumlah:>3}")

Entri hasil crawl : 40
Entri unik        : 26
Kata kunci unik   : 109

10 kata kunci teratas:
  ibl all indonesian                    3
  ibl all indonesian 2026               3
  pt danantara sumberdaya indonesia     3
  basket                                2
  dsi                                   2
  ibl                                   2
  jokowi                                2
  lipsus smartfren                      2
  operator                              2
  portal nadi                           2


## Jawaban Latihan 7

Dari 40 entri hasil crawl H-4 dan Latihan 6, tersisa 26 entri unik setelah judul yang sama dibuang, sesuai dugaan, yaitu 7 news dan 7 inet yang sama pada kedua crawl serta 6 sport dan 6 finance. Dari 26 entri tersebut diperoleh 109 kata kunci unik. Kata kunci terbanyak tidak tunggal, karena tiga kata kunci seri dengan 3 kemunculan, yaitu ibl all indonesian, ibl all indonesian 2026, dan pt danantara sumberdaya indonesia. Kata kunci berikutnya muncul 2 kali, di antaranya basket, dsi, ibl, jokowi, lipsus smartfren, operator, dan portal nadi. Kata kunci seputar IBL diduga berasal dari sitemap sport.detik.com/basket, sedangkan pt danantara sumberdaya indonesia diduga berasal dari sitemap finance, sehingga hasil ini menggambarkan berita pada saat crawling dilakukan dan belum dapat dipakai untuk menyimpulkan topik portal secara umum.

# O. Tugas Mandiri

## Tugas Mandiri. Inverted Index dengan MapReduce

Tugas ini membangun indeks terbalik, yaitu struktur yang menyimpan daftar id judul untuk setiap kata, sehingga pertanyaan seperti judul mana saja yang memuat kata corona dapat dijawab tanpa membaca ulang seluruh judul. Rancangan kode mengikuti rujukan berikut, bukan rancangan sendiri. Dean dan Ghemawat (2004) mendefinisikan fungsi map yang mem-*parse* setiap dokumen dan mengeluarkan pasangan (kata, id dokumen), serta fungsi reduce yang menerima seluruh pasangan sebuah kata, mengurutkan id dokumennya, lalu mengeluarkan pasangan (kata, daftar id dokumen). Tugas Challenge 11 pada mata kuliah CSE 30331 Universitas Notre Dame menambahkan bahwa daftar id pada reducer diurutkan menaik dan dibuat unik. Slide Inverted Indexing dari Jimmy Lin pada mata kuliah CS 451 University of Waterloo menjelaskan versi dasar yang dipakai di sini, yaitu mapper mengeluarkan kata sebagai kunci dan id sebagai nilai, lalu reducer mengumpulkan dan mengurutkan seluruh *posting* sebuah kata.

Penerapannya mengikuti pola H-9, yaitu satu mapper untuk setiap split Parquet, shuffle yang membagi pasangan ke reducer melalui partitioner CRC32 dari H-11, dan reduce yang mengurutkan id unik setiap kata.

Keputusan untuk id kosong pada kategori inet adalah memberi id sintetis, bukan membuang barisnya. Judul inet tetap harus dapat ditemukan melalui indeks, sedangkan pembuangan akan membuat 6,1% judul tidak terjangkau oleh pencarian apa pun. Url juga tidak dapat dipakai sebagai pengganti karena seluruh url inet rusak. Id sintetis dibentuk di dalam mapper dari AWAL_ID_SINTETIS ditambah nomor split dikali RENTANG_ID_SINTETIS_PER_SPLIT dan posisi baris di dalam split, mirip dengan kunci *byte offset* yang dipakai Hadoop untuk menandai baris pada berkas masukan. Cara ini menghasilkan id unik tanpa koordinasi antar-mapper, dan nilainya jauh di atas rentang id artikel detik yang berupa tujuh digit, sehingga posting list tetap diawali id asli dan id sintetis mudah dikenali. Konsekuensinya, id sintetis hanya stabil selama isi dan urutan baris lapisan_terkurasi tidak berubah, sehingga indeks harus dibangun ulang setiap kali lapisan tersebut ditulis ulang.

Rujukan:

1. Dean, J., & Ghemawat, S. (2004). MapReduce: Simplified Data Processing on Large Clusters. OSDI 2004, bagian 2.3. https://www.usenix.org/legacy/event/osdi04/tech/full_papers/dean/dean_html/
2. Lin, J. (2025). CS 451/651 Data-Intensive Distributed Computing, Week 9: Inverted Indexing. University of Waterloo. https://lintool.github.io/cs451-2025f/slides/cs451-2025f-week09b-slides-v1.02.pdf
3. Bui, P. (2016). CSE 30331 Challenge 11: Inverted Index. University of Notre Dame. https://www3.nd.edu/~pbui/teaching/cse.30331.fa16/challenge11.html
4. Python Software Foundation. PEP 8 Style Guide for Python Code dan PEP 257 Docstring Conventions, sebagai acuan penulisan kode. https://peps.python.org/pep-0008/

## Deklarasi Konstanta dan Fungsi Tugas Mandiri

Dilakukan impor terhadap modul collections dan pustaka pandas dengan alias pd. Konstanta JUMLAH_REDUCER_INDEKS bernilai 4 mengikuti R pada H-11, AWAL_ID_SINTETIS bernilai 10.000.000.000, RENTANG_ID_SINTETIS_PER_SPLIT bernilai 1.000.000, *tuple* KATA_CONTOH berisi corona, banjir, dan mudik sebagai tiga kata contoh yang diminta modul, dan JUMLAH_ID_DITAMPILKAN bernilai 5.

Fungsi petakan_indeks(nomor_split, jalur_split) membaca kolom judul dan id satu split, lalu pada *for loop* atas enumerate() setiap baris memakai int(id_detik) apabila id terisi dan id sintetis apabila pd.isna() bernilai True. Fungsi tersebut me-*return* list pasangan (kata, id_dokumen) untuk setiap kata unik judul hasil sorted(set(tokenisasi(judul))), sehingga satu judul hanya menyumbang satu pasangan per kata, beserta jumlah id sintetis yang dibuat pada split tersebut. Fungsi shuffle_indeks(hasil_map, jumlah_reducer) menyiapkan satu defaultdict(list) per reducer dan menambahkan setiap id ke reducer tujuan hasil partitioner() dari H-11. Fungsi reduce_indeks(kata, daftar_id) me-*return* kata beserta sorted(set(daftar_id)), sedangkan jalankan_reducer_indeks(partisi_reducer) menjalankan reduce_indeks() pada setiap kata dalam urutan sorted() di setiap reducer lalu me-*return* *dictionary* indeks.

In [35]:
import collections

import pandas as pd

JUMLAH_REDUCER_INDEKS = 4
AWAL_ID_SINTETIS = 10_000_000_000
RENTANG_ID_SINTETIS_PER_SPLIT = 1_000_000
KATA_CONTOH = ("corona", "banjir", "mudik")
JUMLAH_ID_DITAMPILKAN = 5


def petakan_indeks(nomor_split, jalur_split):
    """Hasilkan pasangan (kata, id_dokumen) satu split.

    Judul tanpa id detik diberi id sintetis dari nomor split dan posisi
    baris, sehingga id tetap unik tanpa koordinasi antar-mapper. Jumlah
    id sintetis yang dibuat dikembalikan bersama pasangannya.
    """
    data_split = pd.read_parquet(jalur_split, columns=["judul", "id"])
    daftar_pasangan = []
    jumlah_id_sintetis = 0
    for posisi_baris, (judul, id_detik) in enumerate(
            zip(data_split["judul"], data_split["id"])):
        if pd.isna(id_detik):
            id_dokumen = (AWAL_ID_SINTETIS
                          + nomor_split * RENTANG_ID_SINTETIS_PER_SPLIT
                          + posisi_baris)
            jumlah_id_sintetis += 1
        else:
            id_dokumen = int(id_detik)
        for kata in sorted(set(tokenisasi(judul))):
            daftar_pasangan.append((kata, id_dokumen))
    return daftar_pasangan, jumlah_id_sintetis


def shuffle_indeks(hasil_map, jumlah_reducer):
    """Kirim setiap pasangan ke reducer tujuan menurut partitioner H-11."""
    partisi_reducer = [
        collections.defaultdict(list) for _ in range(jumlah_reducer)
    ]
    for daftar_pasangan in hasil_map:
        for kata, id_dokumen in daftar_pasangan:
            nomor_reducer = partitioner(kata, jumlah_reducer)
            partisi_reducer[nomor_reducer][kata].append(id_dokumen)
    return partisi_reducer


def reduce_indeks(kata, daftar_id):
    """Kembalikan kata beserta posting list id unik yang terurut."""
    return kata, sorted(set(daftar_id))


def jalankan_reducer_indeks(partisi_reducer):
    """Jalankan reduce pada setiap reducer dengan kunci terurut."""
    indeks = {}
    for partisi in partisi_reducer:
        for kata in sorted(partisi):
            kata_hasil, posting_list = reduce_indeks(kata, partisi[kata])
            indeks[kata_hasil] = posting_list
    return indeks

## Tugas Mandiri. Tahap Map: Pasangan (kata, id_dokumen) per Split

List kosong hasil_map_indeks disiapkan dan variabel jumlah_id_sintetis_total bernilai 0, lalu dijalankan *for loop* atas enumerate(splits) sehingga setiap split memperoleh nomor urut yang sama pada setiap eksekusi karena splits sudah diurutkan pada H-9. Pada setiap iterasi, petakan_indeks() mengembalikan pasangan dan jumlah id sintetis split tersebut, lalu pasangan ditambahkan ke hasil_map_indeks sebagai satu list per mapper dan jumlah id sintetis ditambahkan ke jumlah_id_sintetis_total. Dicetak jumlah id sintetis dan jumlah seluruh pasangan dari mapper.

Dugaan: id sintetis diperkirakan berjumlah 5.577, yaitu seluruh judul inet. Jumlah pasangan diperkirakan sedikit di bawah 825.601 pada H-9, karena kata yang muncul lebih dari sekali pada satu judul hanya dihitung satu kali.

In [36]:
hasil_map_indeks = []
jumlah_id_sintetis_total = 0
for nomor_split, jalur_split in enumerate(splits):
    pasangan_split, jumlah_id_sintetis = petakan_indeks(
        nomor_split, jalur_split
    )
    hasil_map_indeks.append(pasangan_split)
    jumlah_id_sintetis_total += jumlah_id_sintetis

print(f"Judul ber-id sintetis: {jumlah_id_sintetis_total:,}")
print(f"Pasangan dari mapper : "
      f"{sum(len(daftar) for daftar in hasil_map_indeks):,}")

Judul ber-id sintetis: 5,577
Pasangan dari mapper : 817,993


## Tugas Mandiri. Tahap Shuffle dan Reduce

Fungsi shuffle_indeks() membagi hasil_map_indeks ke JUMLAH_REDUCER_INDEKS reducer dan hasilnya di-*assign* ke partisi_reducer, lalu jalankan_reducer_indeks() membentuk *dictionary* indeks_terbalik. Dicetak jumlah kata unik yang menjadi kunci indeks dan total *posting* dari seluruh posting list.

Dugaan: jumlah kata unik diperkirakan sama dengan 32.275 kata pada H-9, dan total *posting* diperkirakan sama dengan jumlah pasangan dari mapper karena setiap pasangan sudah unik per judul.

In [37]:
partisi_reducer = shuffle_indeks(hasil_map_indeks, JUMLAH_REDUCER_INDEKS)
indeks_terbalik = jalankan_reducer_indeks(partisi_reducer)

print(f"Kata unik (kunci): {len(indeks_terbalik):,}")
print(f"Total posting    : "
      f"{sum(len(posting) for posting in indeks_terbalik.values()):,}")

Kata unik (kunci): 32,275
Total posting    : 817,993


## Tugas Mandiri. Tiga Kata Contoh beserta Lima Id Pertama

Dijalankan *for loop* atas KATA_CONTOH, lalu posting list setiap kata diambil dari indeks_terbalik melalui get() berargumen list kosong apabila kata tidak ada. Dicetak setiap kata contoh beserta jumlah judul yang memuatnya dan JUMLAH_ID_DITAMPILKAN id pertamanya melalui *slicing*.

Dugaan: ketiga kata diperkirakan ada di indeks, dengan corona memiliki posting list terpanjang di antara ketiganya. Lima id pertama setiap kata diperkirakan berupa id artikel detik tujuh digit, karena id sintetis berada di ujung posting list.

In [38]:
for kata in KATA_CONTOH:
    posting_list = indeks_terbalik.get(kata, [])
    print(f"{kata:<8} ({len(posting_list):,} judul): "
          f"{posting_list[:JUMLAH_ID_DITAMPILKAN]}")

corona   (17,228 judul): [4856495, 4856598, 4868317, 4868545, 4869163]
banjir   (1,992 judul): [4842198, 4842285, 4842286, 4842287, 4842292]
mudik    (888 judul): [4852294, 4853938, 4856814, 4856929, 4857196]


## Pembahasan Tugas Mandiri

Berdasarkan keluaran di atas, indeks terbalik memuat 32.275 kata, sama dengan jumlah kata unik pada H-9, dengan total 817.993 *posting* dari 90.818 judul, dan 5.577 judul inet memperoleh id sintetis. Jumlah *posting* lebih kecil 7.608 daripada 825.601 pasangan word count pada H-9, karena kata yang berulang di dalam satu judul hanya dicatat satu kali.

Tiga kata contoh menghasilkan posting list berikut.

| Kata | Jumlah judul | Lima id pertama |
|---|---|---|
| corona | 17.228 | 4856495, 4856598, 4868317, 4868545, 4869163 |
| banjir | 1.992 | 4842198, 4842285, 4842286, 4842287, 4842292 |
| mudik | 888 | 4852294, 4853938, 4856814, 4856929, 4857196 |

Seluruh id yang ditampilkan merupakan id artikel detik, karena id sintetis bernilai di atas 10 miliar sehingga selalu berada di ujung posting list. Kata corona dimuat 17.228 judul, sedikit di bawah 17.239 kemunculannya pada H-9, sehingga hanya sedikit judul yang menyebut corona lebih dari sekali.

# Penyimpanan Hasil ke Google Drive

## Mengarsipkan Tiga Lapisan Data ke Folder Praktikum5

Dijalankan *for loop* atas LAPISAN_DIARSIPKAN, yaitu lapisan_mentah, lapisan_terkurasi, dan siap_pakai, lalu setiap folder dikompres ke DIR_SIMPAN melalui arsipkan_lapisan() yang didefinisikan pada sel Menghubungkan Google Drive. Pemanggilan dibungkus blok *try-except*, dan *exception* OSError, misalnya karena folder belum terbentuk atau Drive terputus, ditangani dengan mencetak penanda [GAGAL] beserta pesannya, sedangkan blok *else* mencetak penanda [OK], nama berkas zip, dan ukurannya dalam MB. Terakhir, seluruh isi DIR_SIMPAN dicetak sebagai daftar artefak yang sudah tersimpan permanen.

Keputusan pada sel ini adalah pengarsipan lapisan_mentah secara utuh, karena ukurannya hanya puluhan MB sehingga masih wajar dikumpulkan dan seluruh hasil H-5 dapat diperiksa ulang. Sel ini diletakkan paling akhir agar siap_pakai sudah memuat hasil H-13 sebelum diarsipkan.

In [39]:
for nama_folder in LAPISAN_DIARSIPKAN:
    try:
        jalur_zip = arsipkan_lapisan(nama_folder)
    except OSError as galat:
        print(f"[GAGAL] {nama_folder}: {type(galat).__name__} - {galat}")
    else:
        ukuran_mb = jalur_zip.stat().st_size / 1e6
        print(f"[OK] {nama_folder:<18} -> {jalur_zip.name} "
              f"({ukuran_mb:.1f} MB)")

print("\nIsi DIR_SIMPAN:")
for item in sorted(DIR_SIMPAN.iterdir()):
    print(f"  {item.name}")

[OK] lapisan_mentah     -> lapisan_mentah.zip (5.1 MB)
[OK] lapisan_terkurasi  -> lapisan_terkurasi.zip (7.2 MB)
[OK] siap_pakai         -> siap_pakai.zip (1.2 MB)

Isi DIR_SIMPAN:
  lapisan_mentah.zip
  lapisan_terkurasi.zip
  siap_pakai.zip
